# 트랜스포머

  

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

## Transformer 모델 구조

Transformer는 Seq2seq와 비슷하게 기계번역을 해결하기 위한 인코더와 디코더구조를 가지고 있습니다. Seq2seq와는 달리 인코더 와 디코더 내부에는 MultiHeadAttention 블록과 FeedForwaed라는 블록으로 구성

<center><img src="https://arxiv.org/html/1706.03762v7/extracted/1706.03762v7/Figures/ModalNet-21.png" width="300"/></center>

출처: `https://arxiv.org/abs/1706.03762`


알겠습니다 승완님 🙂  
아래는 말씀하신 **Transformer 구현 단계별 가이드**를 보강해서, 바로 붙여넣기 가능한 **Markdown 버전**입니다. 기존 구조는 유지하면서 **Dropout, Label Smoothing, Learning Rate Scheduler** 같은 실제 구현에서 중요한 요소들을 추가했습니다.  

---

# 🧩 Transformer 구현 단계별 가이드 (실전 보강판)

---

## ⚙️ 1단계: *Scaled Dot-Product Attention*
트랜스포머의 심장인 **Attention(Q, K, V)** 공식을 코드로 직접 구현합니다.

\[
Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

- Query(Q), Key(K), Value(V) 벡터 간 유사도 계산  
- Softmax로 중요도를 구하고, Value에 가중합  
- **Dropout**: Attention 가중치에 Dropout 적용 → 과적합 방지  

---

## 🧠 2단계: *Multi-Head Attention*
1단계에서 만든 어텐션을 여러 개 병렬로 실행하여 “다양한 관점”에서 문장을 바라볼 수 있도록 확장합니다.

- 여러 헤드(`num_heads`)를 통해 병렬 어텐션 수행  
- 각 헤드의 출력을 결합(concat) 후 선형 변환  
- **Residual Connection + LayerNorm + Dropout** 적용  

---

## 🛡️ 3단계: *Padding & Look-Ahead Masks*
어텐션이 불필요한 `<pad>` 토큰을 무시하고, 디코더가 미래 단어를 **미리 엿보지 못하게** 합니다.

- **Padding Mask** → `<pad>` 위치를 가려 계산 제외  
- **Look-Ahead Mask** → 디코더가 다음 단어를 보지 못하도록 제한  

---

## 🏗️ 4단계: 인코더 유닛 조립 — *EncoderLayer*
**MultiHeadAttention + FeedForwardNetwork + Add & Norm**

- Self-Attention → FFN  
- Residual Connection + LayerNorm + Dropout  
- 인코더의 기본 레이어 구조 완성  

---

## 🧩 5단계: 디코더 유닛 조립 — *DecoderLayer*
인코더보다 복잡한 구조를 가집니다.

1. Masked Multi-Head Attention (자기 어텐션)  
2. Encoder–Decoder Attention (인코더 출력과의 어텐션)  
3. Feed Forward Network  

각 부분마다 **Dropout, Residual, LayerNorm**이 적용됩니다.  

---

## 🧱 6단계: 최종 모델 조립 — *Encoder, Decoder, Transformer*
4, 5단계에서 만든 층을 N개씩 쌓고, **Positional Encoding**을 추가해 순서 정보를 부여합니다.

- Encoder: 입력 문장 인코딩  
- Decoder: 타깃 문장 생성  
- Transformer: Encoder + Decoder 결합 모델 완성  

---

## 🗂️ 7단계: 데이터 준비 — *Tokenizer & Dataset*
한국어 챗봇 데이터를 불러와 SentencePiece 토크나이저 학습

- SentencePiece로 단어 분절  
- 텍스트를 정수 시퀀스로 변환  
- Padding 적용 → `PyTorch Dataset` 구성  

---

## 🧮 8단계: 모델 학습 — *Training Loop*
Transformer 학습을 위한 학습 루프 구성

- **모델**: Transformer  
- **Optimizer**: Adam  
- **Loss Function**: CrossEntropyLoss(ignore_index=pad_token)  
- **Label Smoothing**: 정답 분포를 약간 평활화 → 일반화 성능 향상  
- **Learning Rate Scheduler (Warmup)**: 초반에는 학습률을 점진적으로 증가시킨 후 감소  
- **Dropout**: Attention, FFN, Residual 연결마다 적용  
- **훈련 반복**: forward → loss → backward → optimizer.step()  

---

## 🧪 9단계: 추론 및 평가 — *Generation & BLEU*
학습된 모델이 문장을 **스스로 생성(Generate)** 하도록 한 후, 실제 정답과 비교하여 BLEU 점수로 품질을 평가합니다.

- `generate()` 함수로 번역 문장 생성  
- **BLEU Score**: 생성된 문장과 정답 문장의 n-gram 유사도 평가  

---

✅ 이렇게 수정하면, 단순 개념 설명이 아니라 **실제 구현에서 반드시 고려해야 하는 요소들**(Dropout, Label Smoothing, Scheduler 등)이 포함된 **실전용 Transformer 가이드**가 됩니다.  

---


### Scaled Dot-Product Attention

먼저 MultiHeadAttention이 구성되기 위해서는 내부적으로 Scaled Dot-Product Attention 연산이 진행이 되어야 함.  

Scaled Dot-Product Attention는 Query와 Key를 **단순 내적**하여 스코어를 계산합니다.

<center><img src="https://arxiv.org/html/1706.03762v7/extracted/1706.03762v7/Figures/ModalNet-19.png" width="200"/></center>

출처: `https://arxiv.org/abs/1706.03762`


$${Attention}(Q, K, V) = \text{softmax}\Bigl(\frac{Q K^\top}{\sqrt{d_k}}\Bigr) \, V$$

- Query와 Key 간의 단순 내적을 활용하므로, 대규모 병렬 연산이 가능하고 시퀀스 전역에 걸쳐 빠르게 Attention 계산을 수행
- 차원이 커지는 문제를 스케일링(√d_k)으로 보정함으로써, 안정적으로 학습이 가능

이때 Transformer는 Scaled Dot-Product Attention의  Qurey, Key, Value를 전부 동일한 텍스트(임베딩)을 넣어 문장 내의 모든 단어(토큰) 간 의존 관계를 동시에 학습하는 **Self-Attention** 기법으로 활용합니다.

<center><img src="https://drive.google.com/uc?export=view&id=1xpFv-xHu2BnFLkaboNybk5KkQ_vgpwO-" width="600"/></center>


In [ ]:
# [전체 흐름]
# 이 코드는 트랜스포머의 '엔진'이자 '심장'인 'Scaled Dot-Product Attention' 함수를 정의합니다.
# 이 함수의 목적은 "문장 속 단어들이 서로 얼마나 연관되어 있는지"를 계산하여,
# "문맥이 반영된 새로운 단어 벡터"를 만들어내는 것입니다.
#
# 이 과정을 3단계로 비유할 수 있습니다:
# - 유사도 계산 (Q·Kᵀ)
#   → “내 질문(Q)이 다른 단어(Key)와 얼마나 비슷한가?”
#   → 점수표(유사도 행렬)를 만듦.
# - 가중치 계산 (Softmax)
#   → “그 점수를 확률로 바꿔서, 어디에 집중할지 결정한다.”
#   → 각 단어(Key)에 대한 집중 정도(가중치)가 나옴.
# - 정보 가중합 (Weighted Sum with V)
#   → “그 집중 정도만큼 다른 단어들의 실제 정보(Value)를 가져와 섞는다.”
#   → 최종적으로 문맥이 반영된 새로운 단어 벡터가 만들어짐.


# (이 코드를 실행하려면 torch, math, F(functional)가 필요하므로 import합니다.)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 'B' = batch_size (한 번에 처리할 문장 수, 예: 3)
# 'S' = seq_len (문장 길이, 예: 20)
# 'D' = depth/dim (단어 벡터의 차원, 예: 64)
# 'H' = n_heads (멀티 헤드 어텐션의 헤드 수)

def scaled_dot_product_attention(q, k, v, mask=None):
    """
    'Scaled Dot-Product Attention' (어텐션 엔진) 함수를 정의합니다.

    Args:
        q (Query): "질문지"입니다. (예: "I"라는 단어가 "나는 누구와 연관있지?"라고 질문함)
        k (Key): "명패"입니다. (예: "I", "am", "student"가 "나 이런 단어야"라고 들고 있는 명패)
        v (Value): "실제 정보"입니다. (예: "I", "am", "student"가 가진 '진짜' 문맥/의미 정보)
        mask: 특정 위치를 가리는 데 사용 (패딩이나 미래 단어를 무시하기 위해)

    Returns:
        output: 문맥이 반영된 새로운 단어 벡터들
        attention_weights: 각 단어가 다른 단어에 부여한 중요도 점수

    '멀티헤드 어텐션'에서 사용될 경우 (이 코드의 '테스트'에서는 사용되지 않는 경우임)
    q, k, v : shape = (B, H, S, D/H)
    mask    : shape = (B, 1, 1, S) 또는 (B, 1, S, S)
    """
    # --- 1단계: 유사도(Score) 계산 - "질문과 명패 비교하기" ---
    # Query와 Key의 내적(Dot Product)을 계산하여 유사도를 측정합니다.
    # 이는 "내 질문(Q)이 다른 단어들의 명패(K)와 얼마나 일치하는가?"를 수치화한 것입니다.
    #
    # q shape = (배치크기, 문장길이, 임베딩차원) -> (3, 20, 64)
    # k shape = (배치크기, 문장길이, 임베딩차원) -> (3, 20, 64)
    # k.transpose(-1, -2): k의 마지막 두 차원을 뒤집습니다 -> (B, D, S) -> (3, 64, 20)
    #
    # 행렬 곱셈을 수행하여 각 Query와 모든 Key 간의 유사도를 계산합니다.
    # (3, 20, 64) @ (3, 64, 20) = (3, 20, 20)
    # 결과 행렬의 (i, j) 위치는 i번째 단어(Query)와 j번째 단어(Key) 간의 유사도를 나타냅니다.
    matmul_qk = torch.matmul(q, k.transpose(-1, -2))  # Matrix Multiplication (행렬 곱셈)

    # [구체적인 예시로 이해하기]
    # 문장: "I am a student" (3개 단어로 단순화)
    # Query 벡터들:
    #   Q1 = [1, 2]   # "I"의 Query (나는 어떤 단어와 관련이 있을까?)
    #   Q2 = [0, 1]   # "am"의 Query
    #   Q3 = [1, 0]   # "student"의 Query
    #
    # Key 벡터들:
    #   K1 = [1, 0]   # "I"의 Key (나는 이런 특징을 가진 단어야)
    #   K2 = [0, 1]   # "am"의 Key
    #   K3 = [1, 1]   # "student"의 Key
    #
    # 유사도 계산 (Q · K^T):
    #   Q1·K1 = 1*1 + 2*0 = 1   Q1·K2 = 1*0 + 2*1 = 2   Q1·K3 = 1*1 + 2*1 = 3
    #   Q2·K1 = 0*1 + 1*0 = 0   Q2·K2 = 0*0 + 1*1 = 1   Q2·K3 = 0*1 + 1*1 = 1
    #   Q3·K1 = 1*1 + 0*0 = 1   Q3·K2 = 1*0 + 0*1 = 0   Q3·K3 = 1*1 + 0*1 = 1
    #
    # 결과 유사도 행렬:
    #   [[1, 2, 3],
    #    [0, 1, 1],
    #    [1, 0, 1]]

    # --- 2단계: 스케일링 (Scaling) - "점수 안정화하기" ---
    # 유사도 점수가 너무 커지는 것을 방지하기 위해 스케일링합니다.

    # dk: Query 벡터의 차원 크기 (예: 64)
    dk = q.size(-1)

    # √dk로 나누어 점수들을 안정화합니다.
    # 이유: 차원(dk)이 클수록 내적 값이 커지는데, 이렇게 되면 softmax 함수에서
    # 기울기 소실 문제가 발생할 수 있습니다. 스케일링으로 이를 방지합니다.
    scaled_attention_logits = matmul_qk / math.sqrt(dk)

    # [예시 계속]
    # dk = 2, √2 ≈ 1.414
    # 스케일링된 점수:
    #   [[0.71, 1.41, 2.12],
    #    [0.00, 0.71, 0.71],
    #    [0.71, 0.00, 0.71]]

    # --- 3단계: 마스크 적용 (Masking) - "가려야 할 부분 가리기" ---
    # 디코더에서 미래 단어를 보지 못하도록 하거나, 패딩 토큰을 무시하기 위해 사용합니다.
    if mask is not None:
        # mask가 False인 위치(가려야 할 위치)를 -10억으로 설정합니다.
        # 이 값은 softmax를 통과하면 0이 되어 해당 위치를 완전히 무시하게 됩니다.
        scaled_attention_logits = scaled_attention_logits.masked_fill(mask == False, float('-1e9'))

        # [마스크의 중요성]
        # 디코더 훈련 시: 미래 단어를 보지 못하게 함 (컨닝 방지)
        # 패딩 처리 시: 의미 없는 패딩 토큰을 무시함

    # --- 4단계: 가중치 계산 (Softmax) - "집중도 확률로 변환하기" ---
    # 유사도 점수를 0~1 사이의 확률값으로 변환하여 각 Key에 대한 집중도를 계산합니다.

    # dim=-1: 각 Query에 대해 모든 Key에 대한 점수들을 softmax 적용
    # 결과는 각 Query가 각 Key에 주목해야 할 '확률'이 됩니다.
    attention_weights = F.softmax(scaled_attention_logits, dim=-1)

    # [예시 계속 - Softmax 계산 과정]
    # 첫 번째 행 [0.71, 1.41, 2.12]에 softmax 적용:
    #   1. 지수 함수 적용: e^0.71≈2.03, e^1.41≈4.10, e^2.12≈8.33
    #   2. 합 계산: 2.03 + 4.10 + 8.33 = 14.46
    #   3. 정규화: 2.03/14.46≈0.14, 4.10/14.46≈0.28, 8.33/14.46≈0.58
    #
    # 최종 attention_weights:
    #   [[0.14, 0.28, 0.58],   # "I"는 I(14%), am(28%), student(58%)에 집중
    #    [0.00, 0.50, 0.50],   # "am"은 am(50%), student(50%)에 집중
    #    [0.50, 0.00, 0.50]]   # "student"는 I(50%), student(50%)에 집중

    # --- 5단계: 가중합 (Weighted Sum) - "정보 취합하기" ---
    # 계산된 집중도(attention_weights)를 Value에 적용하여 최종 문맥 벡터를 생성합니다.

    # (배치크기, 문장길이, 문장길이) @ (배치크기, 문장길이, 임베딩차원)
    # = (배치크기, 문장길이, 임베딩차원)
    output = torch.matmul(attention_weights, v)  # Matrix Multiplication (행렬 곱셈)

    # [예시 계속 - 가중합 계산]
    # Value 벡터들 정의:
    #   V1 = [1, 0]   # "I"의 실제 정보
    #   V2 = [0, 2]   # "am"의 실제 정보
    #   V3 = [1, 1]   # "student"의 실제 정보
    #
    # "I"의 새로운 문맥 벡터 계산:
    #   = 0.14 * [1, 0]   (I 정보 14% 반영)
    #   + 0.28 * [0, 2]   (am 정보 28% 반영)
    #   + 0.58 * [1, 1]   (student 정보 58% 반영)
    #   = [0.14, 0.00] + [0.00, 0.56] + [0.58, 0.58]
    #   = [0.72, 1.14]
    #
    # 원래 "I"는 [1, 0] 이었지만, 문맥(am, student)을 반영하여 [0.72, 1.14]로 업데이트됨

    return output, attention_weights

# --- 테스트: 실제 동작 확인 ---
# 3개의 문장, 각 문장 20개 단어, 각 단어 64차원 벡터로 테스트
x = torch.randn(3, 20, 64)

# Self-Attention 테스트: Q, K, V 모두 동일한 입력(x) 사용
# 이는 각 단어가 같은 문장 내의 다른 모든 단어와의 관계를 학습하는 것입니다.
out, atw = scaled_dot_product_attention(x, x, x, mask=None)

print(f'attention output shape : {out.shape}')  # (3, 20, 64)
print(f'attention weight shape : {atw.shape}')  # (3, 20, 20)

# [출력 결과 해석]
# output shape: (3, 20, 64) - 입력과 동일한 shape
#   - 3개의 문장, 각 문장 20개 단어, 각 단어 64차원 벡터
#   - 하지만 각 벡터는 이제 원본이 아닌 '문맥이 반영된' 새로운 벡터입니다!
#
# attention_weights shape: (3, 20, 20)
#   - 3개의 문장, 각 문장에서 20개 단어(Query)가 20개 단어(Key)에 부여한 중요도
#   - 각 행의 합은 1 (확률 분포)

# [실제 활용 예시]
# 문장: "The animal didn't cross the street because it was too tired"
# "it"이라는 단어가 "animal"과 "street" 중 어떤 것과 관련있는지 학습:
# - it → animal: 0.8
# - it → street: 0.1
# - it → others: 0.1
# 모델은 "it"이 "animal"을 가리킨다는 것을 스스로 학습합니다!

attention output shape : torch.Size([3, 20, 64])
attention weight shape : torch.Size([3, 20, 20])


### MultiHead Attention

Transformer는 Scaled Dot-Product Attention을 모든 임베딩 차원에서 바로 사용하지 않고 임베딩 차원을 특정 개수로 분할하여 Scaled Dot-Product Attention처리를 합니다.  

MultiHead Attention  임베딩 차원을 여러개의 Head로 분할하여 Scaled Dot-Product Attention처리를 할 뿐만 아니라 각 Q,K,V에 선형레이어를 적용하여 학습 가능한 가중치를 할당해주고 Attention 출력 차원에도 선형레이어로 가중치를 제공합니다.

<center><img src="https://drive.google.com/uc?export=view&id=1YjvM96xK4L3TMWeBfVmDQzTjC_cWZCZs" width="800"/></center>

이런 멀테헤드 어텐션을 적용함으로 다음과 같은 특징을 얻습니다.
- Head는 서로 다른 (임베딩)부분 공간에서 Q, K, V를 학습하여 다양한 관점(패턴, 연관성)을 동시에 포착
- 병렬로 Attention을 수행하여 보다 복잡하고 세밀한 관계를 학습
- Head들의 Attention 결과를 결합(Summarize)함으로써 보다 안정적이고 균형 잡힌 정보를 얻음




In [ ]:
# (이 코드를 실행하려면 torch, nn이 필요하므로 import합니다.)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# [전체 흐름]
# 이 코드는 'MultiHeadAttention' (멀티 헤드 어텐션) 클래스를 정의합니다.
# "Scaled Dot-Product Attention"이 '1명의 전문가(엔진)'라면,
# 'MultiHeadAttention'은 "이 전문가 8명(num_heads)으로 구성된 '전문가 팀'"입니다.

# [비유: 전문가 팀이 협업하는 방식]
# 단일 어텐션: 1명의 전문가가 모든 것을 분석 (과부하 가능)
# 멀티 헤드 어텐션: 여러 전문가가 각자 다른 관점에서 분석 후 결과 통합 (더 풍부한 분석)

class MultiHeadAttention(nn.Module):
    """
    멀티 헤드 어텐션: 여러 개의 어텐션 헤드가 병렬로 작동하여 다양한 관점에서 정보를 분석합니다.

    [비유: 연구팀 구성하기]
    - 팀장 1명이 모든 것을 분석(X)
    - 전문가 8명을 고용하여 각자 다른 측면 분석(O)
    - 경제 전문가, 문법 전문가, 의미론 전문가 등 다양하게 구성
    """

    def __init__(self, em_dim, num_heads):
        """
        멀티 헤드 어텐션 팀을 구성합니다.

        Args:
            em_dim: 임베딩 차원 (전체 정보의 크기) - 예: 512
            num_heads: 헤드의 개수 (전문가 수) - 예: 8
        """
        super(MultiHeadAttention, self).__init__()

        # 팀 기본 정보 저장
        self.num_heads = num_heads  # 전문가 수 (예: 8명)
        self.em_dim = em_dim        # 전체 정보 차원 (예: 512차원)

        # [검증: 전문가 수만큼 일을 나눌 수 있는지 확인]
        # 512차원을 8명이 나눠가지려면 512 % 8 == 0 이어야 합니다.
        # 만약 512 % 7 != 0 이면 "공평하게 나눌 수 없어요!" 오류 발생
        assert em_dim % num_heads == 0, "em_dim must be divisible by num_heads"

        # 각 전문가가 담당할 차원 계산
        # 512 // 8 = 64 (각 전문가는 64차원 정보만 분석)
        self.depth = em_dim // num_heads

        # --- 전문가 팀을 위한 도구 준비 ---

        # [비유: 전문가별 분석 도구]
        # wq: Query 변환기 - "질문을 각 전문가의 관점에 맞게 변환"
        # wk: Key 변환기 - "대상 정보를 각 전문가의 관점에 맞게 변환"
        # wv: Value 변환기 - "실제 내용을 각 전문가의 관점에 맞게 변환"
        self.wq = nn.Linear(em_dim, em_dim)  # (512 -> 512)
        self.wk = nn.Linear(em_dim, em_dim)  # (512 -> 512)
        self.wv = nn.Linear(em_dim, em_dim)  # (512 -> 512)

        # [비유: 팀장 - 개별 보고서를 통합하는 최종 결재자]
        self.dense = nn.Linear(em_dim, em_dim)  # (512 -> 512)

    def split_heads(self, x):
        """
        입력 데이터를 여러 헤드(전문가)에게 분배합니다.

        [비유: 연구 보고서를 전문가별로 나눠주기]
        원본: 두꺼운 단일 보고서 (512페이지)
        분배: 8명의 전문가에게 각각 64페이지씩 나눠주기

        Args:
            x: (batch_size, seq_len, em_dim) 형태의 입력 텐서

        Returns:
            (batch_size, num_heads, seq_len, depth) 형태로 변환된 텐서
        """
        batch_size, seq_len, em_dim = x.size()

        # [1단계: 정보를 논리적으로 쪼개기]
        # (32, 20, 512) -> (32, 20, 8, 64)
        # 512차원을 8개의 64차원 조각으로 나눕니다 (실제 복사 X, 보는 방식만 변경)
        x = x.view(batch_size, seq_len, self.num_heads, self.depth)

        # [2단계: 차원 순서 재배치]
        # (32, 20, 8, 64) -> (32, 8, 20, 64)
        # 이유: 어텐션 계산을 위해 헤드 차원을 배치 차원처럼 취급하기 위함
        x = x.permute(0, 2, 1, 3)

        return x

    def forward(self, v, k, q, mask=None):
        """
        멀티 헤드 어텐션의 실제 작동 과정

        [비유: 전문가 팀의 협업 프로세스]
        1. 자료 준비: 원본 데이터를 팀 공용 형식으로 변환
        2. 업무 분배: 각 전문가에게 담당 영역 할당
        3. 병렬 분석: 모든 전문가가 동시에 자기 영역 분석
        4. 보고서 통합: 개별 분석 결과를 하나로 합침
        5. 최종 검토: 팀장이 통합 보고서 최종 검토

        Args:
            v: Value (실제 내용 정보)
            k: Key (검색 대상 정보)
            q: Query (질문 정보)
            mask: 특정 위치 가리기 (선택사항)

        Returns:
            output: 최종 분석 결과
            attention_weights: 각 전문가의 분석 집중도
        """
        batch_size = q.size(0)

        # --- 1단계: 자료 준비 (원본 데이터를 팀 공용 형식으로 변환) ---

        # [비유: 원본 문서를 팀 내부 표준 형식으로 변환]
        # 각 전문가의 분석 스타일에 맞게 Query, Key, Value 변환
        q = self.wq(q)  # (32, 20, 512) -> (32, 20, 512)
        k = self.wk(k)  # (32, 20, 512) -> (32, 20, 512)
        v = self.wv(v)  # (32, 20, 512) -> (32, 20, 512)

        # --- 2단계: 업무 분배 (각 전문가에게 담당 영역 할당) ---

        # [비유: 두꺼운 보고서를 전문가 수만큼 쪼개서 나눠주기]
        q = self.split_heads(q)  # (32, 20, 512) -> (32, 8, 20, 64)
        k = self.split_heads(k)  # (32, 20, 512) -> (32, 8, 20, 64)
        v = self.split_heads(v)  # (32, 20, 512) -> (32, 8, 20, 64)

        # [분배 결과 설명]
        # - 32개: 처리할 문서(문장) 수
        # - 8개: 전문가 수
        # - 20개: 각 문서의 단어 수
        # - 64개: 각 전문가가 분석할 정보의 차원

        # --- 3단계: 병렬 분석 (모든 전문가가 동시에 자기 영역 분석) ---

        # [비유: 8명의 전문가가 각자 회의실에서 동시에 분석 진행]
        # 각 전문가는 자신에게 할당된 64차원 정보만으로 어텐션 계산
        scaled_attention, attention_weights = scaled_dot_product_attention(q, k, v, mask)

        # [분석 결과]
        # scaled_attention: (32, 8, 20, 64) - 각 전문가의 개별 분석 결과
        # attention_weights: (32, 8, 20, 20) - 각 전문가의 집중도 분포

        # --- 4단계: 보고서 통합 (개별 분석 결과를 하나로 합침) ---

        # [4-1: 차원 순서 원래대로 돌리기]
        # (32, 8, 20, 64) -> (32, 20, 8, 64)
        # 이유: 전문가 차원을 다시 정보 차원으로 합치기 위해
        scaled_attention = scaled_attention.permute(0, 2, 1, 3).contiguous()
        # .contiguous(): 메모리에서 연속적으로 재배열 (뷰 변경 전 필수)

        # [4-2: 모든 전문가의 분석 결과 합치기]
        # (32, 20, 8, 64) -> (32, 20, 512)
        # 8명의 전문가 결과를 하나의 512차원 벡터로 결합
        concat_attention = scaled_attention.view(batch_size, -1, self.em_dim)

        # --- 5단계: 최종 검토 (팀장이 통합 보고서 검토) ---

        # [비유: 팀장이 모든 전문가의 의견을 종합하여 최종 결정]
        output = self.dense(concat_attention)  # (32, 20, 512) -> (32, 20, 512)

        # [최종 출력물]
        # output: (32, 20, 512) - 모든 전문가의 지혜가 모인 최종 분석 결과
        # attention_weights: (32, 8, 20, 20) - 각 전문가가 어디에 집중했는지 기록
        return output, attention_weights

# --- 상세 작동 예시를 통한 이해 ---

# [실제 계산 과정 시뮬레이션]
# 가정: 우리는 2개의 문장을 처리하고, 각 문장은 10개의 단어로 구성됨
# 각 단어는 64차원 벡터로 표현됨

# 테스트 데이터 생성: (2, 10, 64)
# - 2개 배치(문장)
# - 10개 단어
# - 64차원 임베딩
x = torch.randn(2, 10, 64)

# 멀티 헤드 어텐션 모델 생성
# - 전체 차원: 64
# - 헤드 수: 2 (2명의 전문가)
mh = MultiHeadAttention(64, 2)

# 셀프 어텐션 수행 (Q, K, V 모두 동일 입력)
out, atw = mh(x, x, x)

# 결과 분석
print(f'attention output shape : {out.shape}')  # (2, 10, 64)
print(f'attention weight shape : {atw.shape}')  # (2, 2, 10, 10)

# [출력 결과 상세 해석]
# output: (2, 10, 64)
# - 2개 문장에 대해
# - 각 10개 단어가
# - 64차원의 '문맥 반영 벡터'로 변환됨
# - 단일 어텐션과 차원은 같지만 내용이 더 풍부함!

# attention_weights: (2, 2, 10, 10)
# - 2개 문장에 대해
# - 2명의 전문가가 각각
# - 10×10 집중도 행렬을 생성
# - 전문가별로 다른 패턴의 집중도를 보임

# [멀티 헤드의 강점: 다양한 관점의 분석]
# 전문가 1: 주로 '문법적 관계'에 집중
#   - "is" -> "running" (동사 일치)
#   - "the" -> "dog" (관계사-명사 관계)
#
# 전문가 2: 주로 '의미적 관계'에 집중
#   - "dog" -> "running" (개가 뛰는 행위)
#   - "park" -> "running" (공원에서 뛰는 장소)

# [차원 변화 과정 정리]
# 단계                    입력 형태                  출력 형태                  설명
# ------------------------------------------------------------------------------------
# 1. 선형 변환        (2, 10, 64)             (2, 10, 64)              팀 공용 형식 변환
# 2. 헤드 분할        (2, 10, 64)             (2, 2, 10, 32)           전문가별 업무 분배
# 3. 어텐션 계산      (2, 2, 10, 32)          (2, 2, 10, 32)           병렬 분석 수행
# 4. 결과 결합        (2, 2, 10, 32)          (2, 10, 64)              개별 보고서 통합
# 5. 최종 선형층      (2, 10, 64)             (2, 10, 64)              팀장 최종 검토

# [실제 언어 처리에서의 이점]
# 문장: "The bank is by the river and has money"
# - 헤드 1: "bank" -> "river" (강가의 은행 - 지리적 의미)
# - 헤드 2: "bank" -> "money" (돈을 보관하는 은행 - 금융적 의미)
# → 단일 어텐션은 한 가지 의미만 포착할 수 있지만,
# → 멀티 헤드는 다양한 의미를 동시에 포착 가능!

attention output shape : torch.Size([2, 10, 64])
attention weight shape : torch.Size([2, 2, 10, 10])


#### MultiHeadAttention 차원 변화

| 단계                             | 연산 / 함수                                       | 텐서 shape                                          | 설명                             |
| ------------------------------ | --------------------------------------------- | ------------------------------------------------- | ------------------------------ |
| ① 입력                           | `x = torch.randn(2, 10, 64)`                  | **(batch=2, seq_len=10, em_dim=64)**              | 2개 문장, 각 문장 길이 10, 임베딩 차원 64   |
| ② 선형 변환                        | `wq(x), wk(x), wv(x)`                         | (2, 10, 64)                                       | 각 토큰을 Query/Key/Value 공간으로 투영  |
| ③ 헤드 분할 (view)                 | `.view(batch, seq_len, num_heads, depth)`     | (2, 10, 2, 32)                                    | 64차원을 2헤드로 분할 (depth = 32)     |
| ④ 축 재배열 (permute)              | `.permute(0, 2, 1, 3)`                        | **(2, 2, 10, 32)**                                | 헤드 차원을 앞으로 이동                  |
| ⑤ Scaled Dot-Product Attention | `scaled_dot_product_attention(q, k, v, mask)` | **출력:** (2, 2, 10, 32)<br>**가중치:** (2, 2, 10, 10) | 헤드별 어텐션 계산 (가중치: Query×Key 관계) |
| ⑥ 축 재배열 (permute back)         | `.permute(0, 2, 1, 3)`                        | (2, 10, 2, 32)                                    | 다시 seq_len 기준으로 정렬             |
| ⑦ 헤드 결합 (view)                 | `.view(batch, seq_len, em_dim)`               | **(2, 10, 64)**                                   | 여러 헤드를 concat하여 원래 임베딩 차원 복원   |
| ⑧ 최종 선형 변환                     | `self.dense(concat_attention)`                | **(2, 10, 64)**                                   | 모든 헤드 결과를 통합해 최종 출력 생성         |


> pytorch에는 멀티헤드어텐션을 구현한 nn.MultiheadAttention 레어가 존재합니다.

### 어텐션 마스크

Transformer는 학습 효율을 위해 크게 두가지의 마스크를 활용하여 연산의 효율을 얻습니다.

- padding mask
    * 토큰 길이(seq_len)에 못 미쳐 패딩이 된 경우 의미 없는 단어가 연산에 적용되지 않게 하기 위한 마스크
    * 인코더와 디코더에서 둘다 사용

- look-ahead mask(미리보기 방지 마스크)   
    * 디코더의 자기회귀(attention) 과정에서 미래 토큰을 참조하지 못하도록 막는 것   
    * 학습 과정에서 디코더가 미래 토큰을 예측하기 위한 학습을 진행 할때 라벨에서 미래 토큰을 마스크 처리하여 지움    
    * 이를 통해 디코더가 반복적인 학습 없이 한번의 병렬 연산으로 처리됨   
    * Causal Mask 용어로 표현하기도 함

In [ ]:
# (이 코드를 실행하려면 'torch' 라이브러리가 필요합니다.)
import torch

import torch

# [전체 흐름]
# 이 코드는 트랜스포머 모델에서 사용되는 두 가지 중요한 마스킹 기법을 구현합니다.
# 마스킹은 모델이 특정 위치의 정보를 "무시하도록" 강제하는 기법입니다.

# [비유: 시험지 가리개]
# - 패딩 마스크: 문제지의 빈 공간을 가려서 쓸모없는 곳에 집중하지 못하게 함
# - 룩어헤드 마스크: 다음 문제를 미리 보지 못하게 가려서 정직하게 풀게 함

def create_padding_mask(seq, pad_token=0):
    """
    패딩 마스크 생성 함수

    [비유: 문제지의 빈칸 가리기]
    - 실제 문제는 보이게 하고, 빈 칸(패딩)은 가려서 집중하지 못하게 함
    - "I am [PAD] [PAD]" 문장에서 "I", "am"만 보고 [PAD]는 무시하도록 만듦

    Args:
        seq: 입력 시퀀스 텐서 (batch_size, seq_len)
        pad_token: 패딩 토큰의 ID (기본값: 0)

    Returns:
        (batch_size, 1, 1, seq_len) 형태의 패딩 마스크
        - 실제 단어 위치: True (볼 수 있음)
        - 패딩 위치: False (볼 수 없음)
    """
    # [1단계: 실제 단어와 패딩 구분하기]
    # seq 텐서에서 pad_token이 아닌 위치를 찾아 True로 표시
    # 패딩이 아닌 곳(진짜 단어)을 True로 표시하라
    # 예: seq = [[1, 2, 3, 0, 0]] -> mask = [[True, True, True, False, False]]
    mask = (seq != pad_token)

    # [2단계: 차원 확장 - 브로드캐스팅 준비]
    # 어텐션 메커니즘에 맞게 차원을 추가합니다.
    # (batch_size, seq_len) -> (batch_size, 1, 1, seq_len)

    # [왜 차원을 확장할까?]
    # - 원본 마스크: (batch_size, seq_len)
    # - 어텐션 스코어: (batch_size, num_heads, seq_len, seq_len)
    # - 브로드캐스팅을 위해 (batch_size, 1, 1, seq_len)으로 만들어
    #   모든 헤드와 모든 Query 위치에 동일한 마스크를 적용할 수 있게 함

    mask = mask.unsqueeze(1)  # (batch_size, 1, seq_len)
    mask = mask.unsqueeze(2)  # (batch_size, 1, 1, seq_len)

    # [최종 결과물]
    # 예: [[[[True, True, True, False, False]]]]
    # - 배치 1개, 1개의 마스크, 1개의 추가 차원, 시퀀스 길이 5
    return mask

def create_look_ahead_mask(size):
    """
    룩어헤드 마스크(미래 정보 방지 마스크) 생성 함수

    [비유: 시험지에서 다음 문제 가리기]
    - 현재 풀고 있는 문제까지만 보이고, 다음 문제는 미리 보지 못하게 가림
    - 디코더가 "컨닝"하지 못하도록 방지 (자기보다 뒤에 있는 단어를 보지 못함)

    Args:
        size: 시퀀스 길이 (마스크의 크기: size x size)

    Returns:
        (size, size) 형태의 하삼각 행렬 마스크
        - 현재 및 과거 단어: True (볼 수 있음)
        - 미래 단어: False (볼 수 없음)
    """
    # [1단계: 모든 연결 허용하는 마스크 생성]
    # size x size 크기의 모든 값이 True인 행렬 생성
    # 예: size=3 -> [[True, True, True],
    #                [True, True, True],
    #                [True, True, True]]
    mask = torch.ones(size, size, dtype=torch.bool)

    # [2단계: 하삼각 행렬로 변환 - 미래 정보 차단]
    # torch.tril() 함수로 하삼각(lower triangular) 부분만 남기고
    # 상삼각(upper triangular, 미래 정보) 부분은 False로 설정

    # [하삼각 행렬이란?]
    # 대각선을 기준으로 아래쪽(자기 자신 및 과거)만 True
    # 위쪽(미래)은 모두 False
    mask = torch.tril(mask)

    # [변환 과정 시각화]
    # Before tril():      After tril():
    # [[T, T, T],        [[T, F, F],
    #  [T, T, T],   ->    [T, T, F],
    #  [T, T, T]]         [T, T, T]]

    # [각 행의 의미]
    # 행 0: [T, F, F] - 0번 단어는 0번 단어만 볼 수 있음 (1,2번 단어는 미래)
    # 행 1: [T, T, F] - 1번 단어는 0,1번 단어만 볼 수 있음 (2번 단어는 미래)
    # 행 2: [T, T, T] - 2번 단어는 모든 단어를 볼 수 있음 (더 이상 미래 없음)

    return mask

# --- 상세 작동 예시를 통한 이해 ---

# [테스트 데이터 생성]
# 2개의 문장, 각 문장 10개의 토큰
# 0은 패딩 토큰을 의미
inp = torch.Tensor([
    [1, 2, 3, 4, 5, 6, 0, 0, 0, 0],  # 문장 1: 실제 단어 6개 + 패딩 4개
    [3, 5, 6, 2, 6, 5, 8, 0, 0, 0]   # 문장 2: 실제 단어 7개 + 패딩 3개
])

print("원본 입력 텐서:")
print(inp)
print()

# [패딩 마스크 생성 및 분석]
print("=== 패딩 마스크 생성 과정 ===")
pad_mask = create_padding_mask(inp)
print(f'패딩 마스크 shape: {pad_mask.shape}')
print(f'패딩 마스크 내용: \n{pad_mask}')
print()

# [패딩 마스크 상세 해석]
# 첫 번째 문장: [1, 2, 3, 4, 5, 6, 0, 0, 0, 0]
# → 실제 단어: 위치 0~5 (1,2,3,4,5,6)
# → 패딩: 위치 6~9 (0,0,0,0)
# 마스크: [T, T, T, T, T, T, F, F, F, F]

# 두 번째 문장: [3, 5, 6, 2, 6, 5, 8, 0, 0, 0]
# → 실제 단어: 위치 0~6 (3,5,6,2,6,5,8)
# → 패딩: 위치 7~9 (0,0,0)
# 마스크: [T, T, T, T, T, T, T, F, F, F]

# [룩어헤드 마스크 생성 및 분석]
print("=== 룩어헤드 마스크 생성 과정 ===")
lh_mask = create_look_ahead_mask(10)
print(f'룩어헤드 마스크 shape: {lh_mask.shape}')
print(f'룩어헤드 마스크 내용: \n{lh_mask}')
print()

# [룩어헤드 마스크 상세 해석]
# 시퀀스 길이 10에 대한 마스크:
# 행 0: [T, F, F, F, F, F, F, F, F, F] - 0번 단어는 자기 자신만 봄
# 행 1: [T, T, F, F, F, F, F, F, F, F] - 1번 단어는 0,1번 단어만 봄
# 행 2: [T, T, T, F, F, F, F, F, F, F] - 2번 단어는 0,1,2번 단어만 봄
# ...
# 행 9: [T, T, T, T, T, T, T, T, T, T] - 9번 단어는 모든 단어를 봄

# [마스킹의 실제 효과 시뮬레이션]
print("=== 마스킹 효과 시뮬레이션 ===")

# 가상의 어텐션 스코어 생성 (2배치, 1헤드, 10시퀀스, 10시퀀스)
attention_scores = torch.randn(2, 1, 10, 10)
print(f"원본 어텐션 스코어 shape: {attention_scores.shape}")

# 패딩 마스크 적용 효과
print("\n[패딩 마스크 적용 전후 비교]")
print("패딩 위치(False)에 -1e9를 넣어 softmax 후 0이 되도록 만듦")

# 룩어헤드 마스크 적용 효과
print("\n[룩어헤드 마스크 적용 전후 비교]")
print("미래 단어 위치(False)에 -1e9를 넣어 softmax 후 0이 되도록 만듦")

# [두 마스크의 조합 (디코더에서 사용)]
print("\n=== 디코더에서의 마스크 조합 ===")
# 디코더에서는 두 마스크를 모두 사용합니다:
# 1. 룩어헤드 마스크: 미래 단어를 보지 못하게 함
# 2. 패딩 마스크: 패딩 토큰을 무시하게 함

# 마스크 조합 예시
combined_mask = lh_mask & pad_mask[0, 0, 0]  # 첫 번째 배치의 패딩 마스크와 조합
print("조합된 마스크 (첫 번째 배치):")
print(combined_mask)

# [실제 트랜스포머에서의 마스킹 활용 시나리오]
print("\n=== 실제 활용 시나리오 ===")

# 시나리오 1: 인코더에서의 마스킹
print("1. 인코더: 패딩 마스크만 사용")
print("   - 패딩 토큰을 무시하고 실제 단어들 간의 관계만 학습")

# 시나리오 2: 디코더에서의 마스킹
print("\n2. 디코더: 패딩 마스크 + 룩어헤드 마스크 사용")
print("   - 패딩 토큰 무시 + 미래 단어 보지 못하게 함")
print("   - 자기 회귀적 생성 시 컨닝 방지")

# 시나리오 3: 인코더-디코더 어텐션에서의 마스킹
print("\n3. 인코더-디코더 어텐션: 패딩 마스크만 사용")
print("   - 디코더가 인코더 출력을 참조할 때 패딩은 무시")

# [마스킹의 중요성]
print("\n=== 마스킹의 중요성 ===")
print("✓ 패딩 마스크: 계산 효율성 ↑, 무의미한 패딩에 대한 학습 방지")
print("✓ 룩어헤드 마스크: 디코더의 자기 회귀 특성 보장, 컨닝 방지")
print("✓ 병렬 처리 가능: 한 번의 forward pass로 전체 시퀀스 처리")

# [차원 변화 정리]
print("\n=== 차원 변화 요약 ===")
print("패딩 마스크:")
print("입력: (batch_size, seq_len) -> 출력: (batch_size, 1, 1, seq_len)")
print("\n룩어헤드 마스크:")
print("입력: size -> 출력: (size, size)")

# [출력 결과 검증]
print("\n=== 생성된 마스크 검증 ===")
print("패딩 마스크 정상 생성:", pad_mask.shape == (2, 1, 1, 10))
print("룩어헤드 마스크 정상 생성:", lh_mask.shape == (10, 10))
print("패딩 마스크 내용이 예상과 일치:",
      pad_mask[0, 0, 0, 6:].sum() == 0)  # 첫 번째 문장의 패딩 부분이 모두 False인지 확인

padding mask : 
tensor([[[[ True,  True,  True,  True,  True,  True, False, False, False, False]]],


        [[[ True,  True,  True,  True,  True,  True,  True, False, False, False]]]])
look ahead mask : 
tensor([[ True, False, False, False, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False, False, False],
        [ True,  True,  True,  True,  True, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  T

> nn.MultiheadAttention 사용시 마스크 처리될 부분이 True가 되어야 하므로 반대로 bool을 구현해야 합니다.

In [ ]:
# --- 테스트 (이전 코드의 'MultiHeadAttention' 클래스(mh)와 'create_look_ahead_mask'(lh_mask)가 필요합니다) ---

# (가정: mh = MultiHeadAttention(64, 2) (D=64, H=2))
# (가정: lh_mask = create_look_ahead_mask(10) (10x10 크기의 하삼각 행렬))

# (B=2, S=10, D=64) 크기의 '가짜' 입력 텐서 'x'를 생성합니다.
# (2개의 문장, 각 문장은 10개의 단어, 각 단어는 64차원 벡터)
x = torch.randn(2, 10, 64)

# 'MultiHeadAttention' (mh)을 'Self-Attention' 모드 (Q, K, V가 모두 'x'로 동일)로 호출합니다.
# (이것이 "트랜스포머 디코더"의 "첫 번째 서브 레이어"인 'Masked Multi-Head Attention'의 실제 동작입니다.)
#
# [핵심]
# 'mask=lh_mask' (Look-Ahead Mask)를 전달합니다.
# (lh_mask): 10x10 크기의 하삼각 행렬 [[True, False, False...], [True, True, False...], ...]
# (역할) 'scaled_dot_product_attention' 내부에서 'scaled_attention_logits' (점수표)의
# '미래' 부분(False 위치)을 '-1e9'(마이너스 무한대)로 만들어서,
# Softmax 통과 시 확률이 '0'이 되도록 '강제'합니다. (컨닝 방지)
out, atw = mh(x, x, x, mask=lh_mask)

# 'out' (최종 문맥 벡터)의 모양을 출력합니다.
# (결과: (2, 10, 64)) (입력과 모양이 동일합니다. 각 단어 벡터가 '과거의 문맥'만 반영하여 '업데이트'되었습니다.)
print(out.shape)

# 'atw' (어텐션 가중치)의 모양은 (B, H, S, S) -> (2, 2, 10, 10) 입니다.
# 'atw[0]': 2개의 배치(문장) 중 '첫 번째 배치(0번 문장)'를 선택합니다.
# 'atw[0][0]': 2개의 헤드(전문가) 중 '첫 번째 헤드(0번 전문가)'를 선택합니다.
# (결과) '0번 문장'의 '0번 헤드'가 계산한 (S, S) -> (10, 10) '가중치 행렬'을 출력합니다.
# 이 행렬은 "각 단어(행, Query)가 다른 단어(열, Key)에 얼마나 집중했는지"를 0~1 사이의 확률로 보여줍니다.
atw[0][0]

# --- 테스트 출력 결과 ---
# torch.Size([2, 10, 64])
#
# (아래 10x10 행렬 분석이 "Masked"의 핵심입니다.)
# tensor([
#         # [1번단어(Query)가 집중한 비율 (Key)]
#         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         # (해석: 1번 단어는 '자기 자신(1번)'만 100% 보고, '미래(2~10번)'는 0% 봅니다. (마스크 성공!))
#
#         # [2번단어(Query)가 집중한 비율 (Key)]
#         [0.6490, 0.3510, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         # (해석: 2번 단어는 '과거(1번)'와 '현재(2번)'만 보고 (총합 1.0), '미래(3~10번)'는 0% 봅니다. (마스크 성공!))
#
#         # [3번단어(Query)가 집중한 비율 (Key)]
#         [0.4061, 0.2664, 0.3276, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         # (해석: 3번 단어는 '과거/현재(1,2,3번)'만 보고, '미래(4~10번)'는 0% 봅니다. (마스크 성공!))
#
#         [0.1988, 0.2802, 0.3023, 0.2187, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1878, 0.2315, 0.2020, 0.1653, 0.2134, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.2259, 0.1274, 0.2039, 0.1682, 0.1530, 0.1216, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1236, 0.1272, 0.0756, 0.1110, 0.2058, 0.1677, 0.1891, 0.0000, 0.0000, 0.0000],
#         [0.0837, 0.1174, 0.1410, 0.1368, 0.1353, 0.1497, 0.0834, 0.1526, 0.0000, 0.0000],
#         [0.1065, 0.0798, 0.0857, 0.0873, 0.1328, 0.2385, 0.1017, 0.1209, 0.0467, 0.0000],
#
#         # [10번단어(Query)가 집중한 비율 (Key)]
#         [0.1393, 0.0915, 0.0663, 0.0895, 0.0988, 0.1138, 0.1209, 0.0550, 0.0595, 0.1655]],
#         # (해석: 10번 단어는 '모든 과거/현재(1~10번)'를 봅니다. '미래'가 없으므로 0이 없습니다.)
#
#         # (grad_fn=<SelectBackward0>): 이 텐서는 '역전파(학습)'에 사용될 계산 기록을 가지고 있다는 의미입니다.)
#         grad_fn=<SelectBackward0>)

torch.Size([2, 10, 64])


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.6490, 0.3510, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.4061, 0.2664, 0.3276, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1988, 0.2802, 0.3023, 0.2187, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1878, 0.2315, 0.2020, 0.1653, 0.2134, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.2259, 0.1274, 0.2039, 0.1682, 0.1530, 0.1216, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1236, 0.1272, 0.0756, 0.1110, 0.2058, 0.1677, 0.1891, 0.0000, 0.0000,
         0.0000],
        [0.0837, 0.1174, 0.1410, 0.1368, 0.1353, 0.1497, 0.0834, 0.1526, 0.0000,
         0.0000],
        [0.1065, 0.0798, 0.0857, 0.0873, 0.1328, 0.2385, 0.1017, 0.1209, 0.0467,
         0.0000],
        [0.1393, 0.0915, 0.0663, 0.0895, 0.0988, 0.1138, 0.1209, 0.0550, 0.0595,
         0.1655]], grad_fn=<

### 인코더 레이어

- Point-wise Feed Forward Network (PFFN)
    * 멀티헤드어텐션 계층만 활용하면 학습할 수 있는 가중치가 너무 작음
    * 선형 레이어와 활성화 함수를 기반으로 각 토큰의 임베딩을 더 풍부하게 변환하여 모델의 표현력을 높이는 역할

- 인코더 레이어
    * 입력 문장에 대한 특성을 학습 하는 레이어로 하나의 멀티헤드어텐션과 PFFN 레이어가 사용
    * 멀티헤드어텐션엔 하나의 문장이 q,k,v로 입력(셀프어텐션)
    * 어텐션 결과는 입력 임베딩과 더하여 LayerNorm를 통과하고 PFFN 레어에 입력
    * 어텐션 결과와 PFFN 결과를 더하여 LayerNorm에 통과하여 최종 출력

> 어텐션 출력를 입력과 더하는 **Skip Connection**(Residual Connection) 과정은 딥러닝에서 층이 깊어질수록 기울기가 소실/폭주하는 문제를 완하하여 학습 안정성을 향상 시킵니다.

In [ ]:
# (이 코드를 실행하려면 'torch.nn'이 'nn'으로 import 되어 있어야 합니다.)
import torch.nn as nn

# [전체 흐름]
# 이 함수는 "Position-wise Feed-Forward Network" (FFN)라는
# '2층짜리 미니 신경망'을 'nn.Sequential'을 이용해 '간편하게' 정의합니다.
#
# [비유: "개인별 심화 학습실"]
# 'Multi-Head Attention' (1차 회의)이 "단체 토론"을 통해
# "너는 쟤랑 관련 있네" (관계)를 파악하는 단계였다면,
#
# 이 FFN은 '각 단어'가 '자기 자리(Position-wise)'로 돌아가
# "방금 토론한 내용(Attention 결과)을 '개인적으로' 깊게 소화(FFN)"하는 단계입니다.
#
# 이 '심화 학습실'은 2개의 방(Linear)과 1개의 '활성기(GELU)'로 구성됩니다.

# em_dim: (예: 512) '입력' 및 '최종 출력' 차원. 모델의 기본 '작업 차원'입니다.
# feed_dim: (예: 2048) '중간' 은닉층의 차원. (보통 em_dim * 4)
def point_wise_feed_forward_network(em_dim, feed_dim):
    """
    Position-wise Feed-Forward Network (위치별 피드포워드 네트워크)

    [비유: 3단계 정보 가공 공장]
    1단계: 확장 - 정보를 더 넓은 공간으로 펼쳐서 세부 분석
    2단계: 활성화 - 비선형 변환으로 복잡한 패턴 학습
    3단계: 압축 - 분석 결과를 원래 차원으로 정리하여 출력

    Args:
        em_dim: 임베딩 차원 (입력/출력 차원) - 예: 512
        feed_dim: 피드포워드 네트워크의 은닉층 차원 - 예: 2048

    Returns:
        nn.Sequential: 3개의 레이어로 구성된 신경망
    """

    # nn.Sequential: 레이어들을 순차적으로 실행하는 컨테이너
    # [비유: 자동화된 3단계 생산라인]
    return nn.Sequential(
        # [1번 방: "확장(Expand)"]
        # 'nn.Linear' (선형 레이어)를 정의합니다.
        # (입력 차원: 512, 출력 차원: 2048)
        # [비유: 현미경으로 세부 분석하기]
        # 512차원 정보를 2048차원으로 확장하여 더 자세히 분석할 수 있게 함
        #
        # (이유?) 512차원의 '압축된' 정보(Attention 결과)를
        # '더 넓은'(2048차원) 공간으로 '펼쳐서'
        # 다음 단계(GELU)가 더 복잡한 관계를 쉽게 학습할 수 있도록 '재료를 준비'합니다.
        # (비유: 512페이지 요약본을 2048페이지 상세본으로 펼칩니다.)
        # - 512차원은 '압축된' 표현으로, 복잡한 패턴을 학습하기에 제한적
        # - 2048차원으로 펼치면 더 풍부한 특징 추출 가능
        # - 더 넓은 공간에서 복잡한 변환 학습 용이
        #
        # [수학적 이해]
        # 입력: x (batch_size, seq_len, 512)
        # 변환: W₁·x + b₁ (512×2048 행렬 연산)
        # 출력: (batch_size, seq_len, 2048)
        nn.Linear(em_dim, feed_dim),

        # [2번 방: "비선형 변환 (GELU)"]
        # 'GELU' (Gaussian Error Linear Unit) 활성화 함수를 통과시킵니다.
        # [비유: 창의적 사고 발휘하기]
        # 선형 변환만으로는 복잡한 패턴 학습 불가 → 비선형성 도입 필요
        #
        # (이유? ⭐️⭐️⭐️ 여기가 '학습'의 핵심 ⭐️⭐️⭐️)
        # 'Linear'만 100번 쌓아도 그건 '하나의 큰 Linear'일 뿐, 복잡한 패턴을 못 배웁니다.
        # 'GELU'같은 '비선형(non-linear)' 함수가 중간에 끼어들어야
        # "이 정보는 중요하니 살리고, 저 정보는 무시하자" 같은
        # '복잡한 의사결정(패턴)'을 학습할 수 있습니다.
        # (비유: 펼쳐진 2048페이지 상세본에서 "중요한 부분만 형광펜"을 칠합니다.)
        #
        # [GELU의 작동 원리]
        # GELU(x) = x * Φ(x) (여기서 Φ(x)는 표준정규분포의 CDF)
        # - ReLU: "0 이하는 0, 0 이상은 그대로"
        # - GELU: "입력값이 클수록 더 많이 통과, 작을수록 더 많이 차단"
        # - 더 부드러운 비선형성으로 그래디언트 흐름 개선
        #
        # [GELU의 직관적 이해]
        # 입력: [1.0, 0.5, -0.5, -1.0]
        # GELU: [0.84, 0.32, -0.15, -0.16] (값에 따라 다른 비율로 통과)
        # ReLU: [1.0, 0.5, 0.0, 0.0] (단순한 이분법)
        #
        # [비선형성의 중요성]
        # 선형 변환만 여러 층 쌓아도 결국 하나의 선형 변환과 동일
        # 비선형 활성화 함수가 있어야 진정한 "심층" 학습 가능
        # "이 정보는 강조, 저 정보는 억제" 같은 복잡한 결정 가능
        nn.GELU(),

        # [3번 방: "압축(Compress)"]
        # 'nn.Linear' (선형 레이어)를 다시 정의합니다.
        # (입력 차원: 2048, 출력 차원: 512)
        # [비유: 분석 결과를 간결한 보고서로 정리]
        # 2048차원의 풍부한 정보를 원래의 512차원 작업 공간으로 압축
        #
        # (이유?) 'GELU'까지 통과한 '풍부해진' 2048차원 정보를
        # 다시 원래의 '작업 차원'(512)으로 '압축'해서 - 다음 트랜스포머 블록이나 레이어가 동일한 차원을 기대
        # 다음 '블록(Nx)'이나 '레이어'로 전달해야 하기 때문입니다. - 차원 일관성 유지를 위해 원래 차원으로 복원
        # (비유: 형광펜 칠한 2048페이지를 '새로운 512페이지 요약본'으로 만듭니다.) - 불필요한 정보를 걸러내고 핵심 정보만 전달
        #
        # [압축의 이점]
        # - 계산 효율성: 낮은 차원에서의 연산 속도 향상
        # - 정보 응집: 가장 중요한 정보만 보존
        # - 차원 호환: Residual Connection과의 덧셈 가능
        nn.Linear(feed_dim, em_dim)
    )

    # --- 상세 작동 예시를 통한 이해 ---

# [FFN의 실제 데이터 흐름 시뮬레이션]
print("=== FFN 데이터 흐름 시뮬레이션 ===")

# 가상의 입력 데이터 생성 (배치 2, 시퀀스 길이 5, 임베딩 512)
# [비유: 2개의 문서, 각 문서 5개 단어, 각 단어 512차원 표현]
batch_size = 2
seq_len = 5
em_dim = 512
feed_dim = 2048  # 일반적으로 em_dim의 4배

# 더미 입력 데이터
dummy_input = torch.randn(batch_size, seq_len, em_dim)
print(f"입력 데이터 형태: {dummy_input.shape}")  # (2, 5, 512)

# FFN 모델 생성
ffn = point_wise_feed_forward_network(em_dim, feed_dim)

# FFN 통과
output = ffn(dummy_input)
print(f"출력 데이터 형태: {output.shape}")  # (2, 5, 512)

# [FFN의 위치별(Position-wise) 특성 설명]
print("\n=== FFN의 위치별(Position-wise) 처리 ===")
print("FFN은 각 단어 위치에서 독립적으로 동일한 연산을 수행합니다")
print("문장: 'I love machine learning'")
print("위치 0 ('I'): FFN 독립 적용 → 새로운 'I' 표현 생성")
print("위치 1 ('love'): FFN 독립 적용 → 새로운 'love' 표현 생성")
print("위치 2 ('machine'): FFN 독립 적용 → 새로운 'machine' 표현 생성")
print("... (모든 위치에서 동일한 FFN 가중치 사용)")

# [FFN과 어텐션의 협력 관계]
print("\n=== FFN과 어텐션의 협력 관계 ===")
print("어텐션: '단어들 간의 관계' 학습 (Who should pay attention to whom?)")
print("FFN: '각 단어의 내부 표현' 풍부하게 학습 (How to process my information?)")

# [차원 변화 상세 분석]
print("\n=== FFN 내부 차원 변화 분석 ===")
print("1. 입력: (2, 5, 512) - 2문장, 5단어, 512차원")
print("2. Linear(512→2048): (2, 5, 2048) - 정보 확장")
print("3. GELU(): (2, 5, 2048) - 비선형 변환")
print("4. Linear(2048→512): (2, 5, 512) - 정보 압축")
print("5. 출력: (2, 5, 512) - 원래 차원 유지")

# [FFN의 실제 학습 패턴 예시]
print("\n=== FFN이 학습하는 패턴 예시 ===")
print("입력 단어: 'bank'")
print("어텐션 후: 'river', 'money', 'financial' 정보 반영된 'bank'")
print("FFN 후: ")
print("- 은행 관련 의미 강조: 'financial', 'money', 'account' 특징 활성화")
print("- 강가 의미 억제: 'water', 'river', 'flow' 특징 감소")

# [다양한 FFN 구성 비교]
print("\n=== 다양한 FFN 구성 ===")
print("표준: 512 → 2048 → 512 (가장 일반적)")
print("축소: 512 → 1024 → 512 (계산량 감소)")
print("확장: 512 → 4096 → 512 (표현력 증가)")

# [활성화 함수 비교]
print("\n=== 활성화 함수 비교 ===")
print("ReLU: 간단하고 빠르지만 음수 정보 완전 손실")
print("GELU: 부드러운 비선형성, BERT/GPT에서 널리 사용")
print("Swish: GELU와 유사하지만 더 간단한 형태")

# [FFN의 실제 언어 처리 예시]
print("\n=== 실제 언어 처리에서의 FFN 역할 ===")

print("\n예시 1: 다의어 처리")
print("입력: 'I went to the bank to deposit money'")
print("FFN이 'bank'를 '금융 기관' 의미로 변환")

print("\n예시 2: 문법 구조 학습")
print("입력: 'The cats are running'")
print("FFN이 'cats'와 'are'의 일치 관계를 인코딩")

print("\n예시 3: 의미론적 관계")
print("입력: 'king - man + woman = queen'")
print("FFN이 벡터 공간에서의 의미론적 연산 수행")

# [FFN의 수학적 표현]
print("\n=== FFN의 수학적 표현 ===")
print("FFN(x) = Linear₂(GELU(Linear₁(x)))")
print("여기서:")
print("Linear₁: W₁·x + b₁  (512 → 2048)")
print("GELU: Gaussian Error Linear Unit")
print("Linear₂: W₂·x + b₂  (2048 → 512)")

# [FFN의 계산 복잡도 분석]
print("\n=== FFN 계산 복잡도 ===")
print("어텐션 복잡도: O(n²·d) - 시퀀스 길이에 민감")
print("FFN 복잡도: O(n·d²) - 임베딩 차원에 민감")
print("∴ 긴 시퀀스: 어텐션 병목, 큰 임베딩: FFN 병목")

# [실제 트랜스포머에서의 FFN 위치]
print("\n=== 트랜스포머 블록 내 FFN 위치 ===")
print("1. 멀티 헤드 어텐션 (관계 분석)")
print("2. Add & Norm (잔차 연결 + 정규화)")
print("3. FFN (개별 표현 풍부화) ← 여기!")
print("4. Add & Norm (잔차 연결 + 정규화)")

# [검증: FFN 출력 특성]
print("\n=== FFN 출력 특성 검증 ===")
print("입력과 출력 차원 동일:", dummy_input.shape == output.shape)
print("FFN 파라미터 수:", sum(p.numel() for p in ffn.parameters()))

# 파라미터 계산 설명
ffn_params = em_dim * feed_dim + feed_dim  # 첫 번째 레이어
ffn_params += feed_dim * em_dim + em_dim   # 두 번째 레이어
print(f"계산: {em_dim}×{feed_dim} + {feed_dim} + {feed_dim}×{em_dim} + {em_dim}")
print(f"총 파라미터: {ffn_params:,}")

print("\n=== FFN의 핵심 요약 ===")
print("✓ 목적: 각 단어의 표현을 풍부하게 하는 개인 학습실")
print("✓ 구조: 확장 → 비선형 변환 → 압축의 3단계")
print("✓ 특징: 위치별 독립 적용, 동일 가중치 공유")
print("✓ 역할: 어텐션의 관계 분석을 보완하는 내부 표현 강화")

In [ ]:
# (이 코드를 실행하려면 'torch'와 'nn'이 import 되어 있어야 하며,
# 'MultiHeadAttention'과 'point_wise_feed_forward_network'가
# 이전 코드에서 정의되어 있어야 합니다.)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# (MultiHeadAttention, point_wise_feed_forward_network 등 이전 코드 정의가 여기에 있다고 가정)
# ... (이전 코드 생략) ...

# [전체 흐름]
# 이 코드는 트랜스포머 '인코더'의 '블록(Block) 1개' (1층)를 구현한 것입니다.
# "원본 문서 분석팀" 비유에서, 이 블록은 "1차 분석팀" (1학년)에 해당합니다.
#
# 이 블록은 2개의 '서브 레이어'(회의실)로 구성됩니다.
# 1. 1차 회의: Multi-Head Attention (서로 '관계'를 파악하는 단체 회의)
# 2. 2차 회의: Feed Forward Network (각자 '개인 학습'으로 소화하는 시간)
# (중요!) 각 회의가 끝날 때마다 'Add & Norm' (보고서 취합)을 수행합니다.
# 각 단계 후에는 드롭아웃으로 과적합 방지 (10% 무작위 정보 삭제)

class EncoderLayer(nn.Module):
    '''
    # (클래스 설명 주석: 이 클래스가 무엇을 하는지 요약)
    # 한 개의 Encoder Layer는 ① Multi-Head Attention → ② Feed Forward Network 두 블록으로 구성
    # 각 블록 뒤에는 Residual Connection(잔차 연결) + Layer Normalization 이 있습니다.
    '''
    # [지금 하는 일: '분석팀 블록' 초기화 함수(생성자) 정의]
    # 'EncoderLayer'라는 '설계도'로 '객체(블록)'를 만들 때 (예: `enc_layer = EncoderLayer(...)`)
    # 이 __init__ 함수가 호출되어, 이 블록에 필요한 '부품(레이어)'들을 '미리' 준비합니다.
    #
    # em_dim: (예: 32) 모델의 기본 '작업 차원' (단어 벡터의 크기)
    # num_heads: (예: 4) Multi-Head Attention에서 사용할 '전문가(헤드)'의 수
    # feed_dim: (예: 64) FFN의 '중간 확장' 차원 (보통 em_dim * 2 또는 * 4)
    # rate: (예: 0.1) 드롭아웃 비율 (훈련 시 10%의 뉴런을 끔)
    #
    def __init__(self, em_dim, num_heads, feed_dim, rate=0.1):
        # 부모 클래스(nn.Module)의 초기화 함수를 호출합니다. (파이토치 모델 정의 시 필수)
        super(EncoderLayer, self).__init__()

        # '1차 회의실' 부품: 'MultiHeadAttention' (전문가 팀)을 생성합니다.
        # (이전 코드에서 정의한 MultiHeadAttention 클래스를 사용합니다.)
        # (예: 입력 차원 32, 전문가 4명으로 설정)
        self.mha = MultiHeadAttention(em_dim, num_heads)

        # '2차 회의실' 부품: 'FeedForwardNetwork' (개인별 심화 학습실)을 생성합니다.
        # (이전 코드에서 정의한 point_wise_feed_forward_network 함수를 사용합니다.)
        # (예: 입력 32 -> 중간 64 -> 출력 32)로 설정
        self.ffn = point_wise_feed_forward_network(em_dim, feed_dim)

        # "보고서 정리" 부품들을 정의합니다.

        # '1차 보고서 정리' 부품: 'Layer Normalization' (포맷 정리 기계 1)을 생성합니다.
        # (em_dim=32) 32차원 벡터의 '숫자 분포'를 정규화합니다.
        # (eps=1e-6): 분모가 0이 되는 것을 방지하기 위한 아주 작은 값입니다.
        self.layernorm1 = nn.LayerNorm(em_dim, eps=1e-6)

        # '2차 보고서 정리' 부품: 'Layer Normalization' (포맷 정리 기계 2)를 생성합니다.
        # (주의: layernorm1과 2는 '서로 다른' 가중치를 가진 '별개의' 부품입니다.)
        self.layernorm2 = nn.LayerNorm(em_dim, eps=1e-6)

        # '과적합 방지' 부품 1: 'Dropout' (10% 확률로 뉴런 비활성화)
        # (MHA 회의 결과에 적용할 예정)
        self.dropout1 = nn.Dropout(rate)

        # '과적합 방지' 부품 2: 'Dropout' (FFN 결과에 적용할 예정)
        self.dropout2 = nn.Dropout(rate)

    # [지금 하는 일: '분석팀 블록'의 실제 동작(순전파) 흐름 정의]
    # '설계도(__init__)'에 따라 만든 '부품'들을 '조립'하여,
    # '데이터가 입력되었을 때(forward)' '어떤 순서로 작동'할지 '흐름'을 정의합니다.
    # x: (B, S, D) (예: [3, 10, 32]) - '페이지 번호가 매겨진 문서' (이 블록의 입력)
    # mask: (선택 사항) '패딩 마스크' (예: [3, 1, 1, 10]) (0([PAD])인 부분을 무시하기 위함)
    def forward(self, x, mask=None):
        # --- 1차 회의 (Multi-Head Attention) + 1차 보고서 취합 (Add & Norm) ---

        # [1. 회의] 'self.mha' (Multi-Head Attention)를 실행합니다.
        # (Q=x, K=x, V=x): 'Self-Attention' 모드. "문서(x)가 자기 자신(x)을 참조하라."
        # (mask): 패딩 마스크를 전달하여 "[PAD]" 토큰은 무시하도록 합니다.
        # (attn_output): 1차 회의 결과 (B, S, D) -> (3, 10, 32)
        # (_): 어텐션 가중치(atw)도 반환되지만, 여기서는 '결과'만 필요하므로 무시합니다.
        attn_output, _ = self.mha(x, x, x, mask)

        # [2. 과적합 방지] 회의 결과(attn_output)에 'self.dropout1'을 적용합니다.
        # (훈련 시에만 10%의 뉴런을 끔으로써, 모델이 특정 뉴런에 과의존하는 것을 방지합니다.)
        attn_output = self.dropout1(attn_output)

        # [3. 보고서 취합 (Add & Norm)]
        # (1) 'x + attn_output' (Add: Residual Connection):
        #     '회의 전 원본(x)'과 '회의 후 결과(attn_output)'를 '더합니다'. (스테이플러 비유)
        #     (이유: 원본 정보를 보존하여 '기울기 소실'을 막고 학습을 안정화합니다.)
        # (2) self.layernorm1(...):
        #     '더해진 결과'를 '1번 정규화 기계(layernorm1)'에 넣어 숫자 분포를 '정리'합니다.
        # 'out1' (1차 회의 최종본) shape: (B, S, D) -> (3, 10, 32)
        out1 = self.layernorm1(x + attn_output)

        # --- 2차 회의 (Feed-Forward Network) + 2차 보고서 취합 (Add & Norm) ---

        # [4. 개인 학습] 'out1' (1차 회의 최종본)을 'self.ffn' (개인 심화 학습실)에 넣습니다.
        # (내부 동작: 32차원 -> 64차원(확장) -> GELU -> 32차원(압축))
        # (모든 단어(S=10)가 '각자' 이 FFN을 통과하여 '더 똑똑한' 벡터로 변환됩니다.)
        # 'ffn_output' (개인 학습 결과) shape: (B, S, D) -> (3, 10, 32)
        ffn_output = self.ffn(out1)

        # [5. 과적합 방지] 개인 학습 결과(ffn_output)에 'self.dropout2'를 적용합니다.
        ffn_output = self.dropout2(ffn_output)

        # [6. 보고서 취합 (Add & Norm)]
        # (1) 'out1 + ffn_output' (Add: Residual Connection):
        #     '개인 학습 전(out1)'과 '개인 학습 후(ffn_output)'를 '더합니다'. (원본 보존)
        # (2) self.layernorm2(...):
        #     '더해진 결과'를 '2번 정규화 기계(layernorm2)'에 넣어 최종 '포맷 정리'를 합니다.
        # 'out2' (이 블록의 최종 출력) shape: (B, S, D) -> (3, 10, 32)
        out2 = self.layernorm2(out1 + ffn_output)

        # 이 '1개 층(블록)'의 모든 분석을 마친 '최종 결과물(out2)'을 반환합니다.
        # (이 'out2'는 다음 층(블록)의 'x' 입력으로 들어가게 됩니다.)
        return out2

# --- 테스트 ---
# '가짜' 입력 데이터 생성: (B=3, S=10, D=32)
# (3개 문장, 각 문장 10개 단어, 각 단어 32차원)
enc_inp = torch.randn(3, 10, 32)

# 'EncoderLayer' 클래스(설계도)로 'enc_layer' (1개 층) 객체를 생성합니다.
# (em_dim=32, num_heads=4, feed_dim=64)
enc_layer = EncoderLayer(32, 4, 64)

# 생성한 'enc_layer' 객체(모델)에 'enc_inp' 데이터를 넣고 'forward' 함수를 실행합니다.
# (mask=None: 패딩 마스크를 사용하지 않음)
enc_out = enc_layer(x=enc_inp, mask=None)

# 'enc_layer'가 최종적으로 반환한 'enc_out' 텐서의 '모양(shape)'을 확인합니다.
# (예상 출력: torch.Size([3, 10, 32]))
# (이유: 트랜스포머 블록은 입력(B, S, D)과 출력(B, S, D)의 모양이 동일하게 유지됩니다.)
enc_out.shape

torch.Size([3, 10, 32])

### 디코더 레이어

디코더 레이어는 예측 문장을 만들기 위해 이전 토큰으로 미래 토큰을 학습 하는 레이어로 2개의 멀티헤드어텐션과 하나의 PFFN을 사용합니다.


    * 첫번째 멀티헤드어텐션: 디코더 입력으로 셀프어텐션 진행
    * 첫번째 LayerNorm: 어텐션 결과와 디코더 입력을 더해 LayerNorm 진행
    * 두번째 멀티헤드어텐션: 인코더의 출력이 V,K로 입력되고 첫번째 LayerNorm 출력이 Q로 입력되어 어텐션 진행
    * 두번째 LayerNorm: 첫번째 LayerNorm 출력과 두번째 어텐션 결과를 더해 LayerNorm 진행
    * PFFN: 두번째 LayerNorm 결과를 입력하여 출력 결과를 더해 LayerNorm 진행

In [ ]:
# (이 코드를 실행하려면 'torch', 'nn'이 import 되어 있어야 하며,
# 'MultiHeadAttention'과 'point_wise_feed_forward_network'가
# 이전 코드에서 정의되어 있어야 합니다.)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# (MultiHeadAttention, point_wise_feed_forward_network 등 이전 코드 정의가 여기에 있다고 가정)
# ... (이전 코드 생략) ...

# [전체 흐름]
# 이 코드는 트랜스포머 '디코더'의 '블록(Block) 1개' (1층)를 구현한 것입니다.
# "단어 생성 작가팀" 비유에서, 이 블록은 "작가팀의 1학년" 과정에 해당합니다.
#
# 인코더 블록과 달리, 디코더 블록은 3개의 '서브 레이어'(회의실)로 구성됩니다.
# 1. 1차 회의: Masked Multi-Head Self-Attention ("지금까지 쓴 초고"를 '컨닝 없이' 검토)
# 2. 2차 회의: Encoder-Decoder Attention ("원본 책(인코더)"을 '참조')
# 3. 3차 회의: Feed Forward Network ("모든 정보"를 '개인적으로 소화')
#
# (중요!) 각 회의가 끝날 때마다 'Add & Norm' (보고서 취합)을 수행합니다.
class DecoderLayer(nn.Module):
    '''
    # (클래스 설명 주석: 이 클래스가 무엇을 하는지 요약)
    # 3개의 서브층(sub-layer)
    #   1) Masked Multi-Head Self-Attention: 디코더 내부 단어들끼리 어텐션 (미래 단어 가리기)
    #   2) Encoder–Decoder Attention: 인코더의 출력과 어텐션 (입력 문장 참고)
    #   3) Feed Forward Network (FFN): 각 단어 벡터 독립적으로 비선형 변환
    '''
    # [지금 하는 일: '작가팀 블록' 초기화 함수(생성자) 정의]
    # 'DecoderLayer'라는 '설계도'로 '객체(블록)'를 만들 때
    # 이 __init__ 함수가 호출되어, 이 블록에 필요한 '부품(레이어)'들을 '미리' 준비합니다.
    # em_dim: (예: 32) 모델의 기본 '작업 차원' (단어 벡터의 크기)
    # num_heads: (예: 4) Multi-Head Attention에서 사용할 '전문가(헤드)'의 수
    # feed_dim: (예: 64) FFN의 '중간 확장' 차원 (보통 em_dim * 2 또는 * 4)
    # rate: (예: 0.1) 드롭아웃 비율 (훈련 시 10%의 뉴런을 끔)
    def __init__(self, em_dim, num_heads, feed_dim, rate=0.1):
        # 부모 클래스(nn.Module)의 초기화 함수를 호출합니다. (파이토치 모델 정의 시 필수)
        super(DecoderLayer, self).__init__()

        # '1차 회의실' 부품: 'Masked Self-Attention'용 MultiHeadAttention 레이어를 생성합니다.
        # (이름: self.mha1)
        self.mha1 = MultiHeadAttention(em_dim, num_heads)

        # '2차 회의실' 부품: 'Encoder-Decoder Attention'용 MultiHeadAttention 레이어를 생성합니다.
        # (이름: self.mha2)
        # (중요!) 'mha1'과 'mha2'는 '서로 다른' 가중치를 가진 '별개의' 부품(레이어)입니다.
        self.mha2 = MultiHeadAttention(em_dim, num_heads)

        # '3차 회의실' 부품: 'FeedForwardNetwork' (개인별 심화 학습실)을 생성합니다.
        self.ffn = point_wise_feed_forward_network(em_dim, feed_dim)

        # "보고서 정리" 부품들을 정의합니다. (3개의 서브 레이어이므로 3개씩 필요)

        # '1차 보고서 정리' 부품: 'Layer Normalization' (포맷 정리 기계 1)
        self.layernorm1 = nn.LayerNorm(em_dim, eps=1e-6)
        # '2차 보고서 정리' 부품: 'Layer Normalization' (포맷 정리 기계 2)
        self.layernorm2 = nn.LayerNorm(em_dim, eps=1e-6)
        # '3차 보고서 정리' 부품: 'Layer Normalization' (포맷 정리 기계 3)
        self.layernorm3 = nn.LayerNorm(em_dim, eps=1e-6)

        # '과적합 방지' 부품 1: 'Dropout' (mha1 결과에 적용)
        self.dropout1 = nn.Dropout(rate)
        # '과적합 방지' 부품 2: 'Dropout' (mha2 결과에 적용)
        self.dropout2 = nn.Dropout(rate)
        # '과적합 방지' 부품 3: 'Dropout' (ffn 결과에 적용)
        self.dropout3 = nn.Dropout(rate)

    # [지금 하는 일: '작가팀 블록'의 실제 동작(순전파) 흐름 정의]
    # '데이터가 입력되었을 때(forward)' '어떤 순서로 작동'할지 '흐름'을 정의합니다.
    # [입력값]
    # x: (B, S_dec, D) - '디코더'의 입력. (예: "지금까지 쓴 초고" [BOS], "나는", "학생")
    # enc_output: (B, S_enc, D) - '인코더'의 '최종 분석본'. (예: "I", "am", "student" 정보)
    # look_ahead_mask: (S_dec, S_dec) - 1차 회의(mha1)를 위한 '컨닝 방지' 마스크
    # padding_mask: (B, 1, 1, S_enc) - 2차 회의(mha2)를 위한 '인코더 패딩 무시' 마스크
    def forward(self, x, enc_output, look_ahead_mask=None, padding_mask=None):

        # --- 1차 회의 (Masked Multi-Head Self-Attention) + 1차 보고서 취합 (Add & Norm) ---
        # (목적: "지금까지 쓴 초고(x)"를 '컨닝 없이' 스스로 검토하기)

        # [1. 회의] 'self.mha1' (1차 회의실)을 실행합니다.
        # (Q=x, K=x, V=x): 'Self-Attention' 모드. "초고(x)가 자기 자신(x)을 참조하라."
        # (look_ahead_mask): '미래 단어 훔쳐보기'를 방지하는 '가면'을 씁니다. (매우 중요!)
        # (attn1): 1차 회의 결과 (B, S_dec, D)
        # (attn_weights_block1): 1차 회의 가중치 (B, H, S_dec, S_dec) (어디에 집중했는지 기록)
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)

        # [2. 과적합 방지] 회의 결과(attn1)에 'self.dropout1'을 적용합니다.
        attn1 = self.dropout1(attn1)

        # [3. 보고서 취합 (Add & Norm)]
        # (1) 'x + attn1' (Add: Residual Connection):
        #     '초고 검토 전 원본(x)'과 '초고 검토 후 결과(attn1)'를 '더합니다'. (원본 보존)
        # (2) self.layernorm1(...):
        #     '더해진 결과'를 '1번 정규화 기계(layernorm1)'에 넣어 숫자 분포를 '정리'합니다.
        # 'out1' (1차 회의 최종본) shape: (B, S_dec, D)
        out1 = self.layernorm1(x + attn1)

        # --- 2차 회의 (Encoder-Decoder Attention) + 2차 보고서 취합 (Add & Norm) ---
        # (목적: "원본 책(enc_output)"을 '참조'하여, "초고(out1)"에 필요한 정보 가져오기)

        # [4. 회의] 'self.mha2' (2차 회의실)을 실행합니다. (⭐️⭐️⭐️ 여기가 핵심 ⭐️⭐️⭐️)
        # [Q, K, V의 정체]
        # Q (Query) = out1  : '디코더'의 "1차 회의 최종본"이 '질문지'가 됩니다.
        #                   (의미: "나 지금 이 초고(out1) 쓰는데, 원본 책에서 뭘 참고할까?")
        # K (Key)   = enc_output : '인코더'의 "최종 분석본"이 '원본 책 인덱스'가 됩니다.
        # V (Value) = enc_output : '인코더'의 "최종 분석본"이 '원본 책 내용'이 됩니다.
        #
        # (padding_mask): "원본 책(enc_output)"에 [PAD]가 있다면, 그건 "참조하지 말라"고 알려줍니다.
        # (attn2): 2차 회의 결과 (B, S_dec, D)
        # (attn_weights_block2): 2차 회의 가중치 (B, H, S_dec, S_enc) (디코더가 인코더의 '어디'에 집중했는지 기록)
        attn2, attn_weights_block2 = self.mha2(v=enc_output, k=enc_output, q=out1, mask=padding_mask)

        # [5. 과적합 방지] 2차 회의 결과(attn2)에 'self.dropout2'를 적용합니다.
        attn2 = self.dropout2(attn2)

        # [6. 보고서 취합 (Add & Norm)]
        # (1) 'out1 + attn2' (Add: Residual Connection):
        #     '원본 참조 전(out1)' 상태와 '원본 참조 후(attn2)' 결과를 '더합니다'. (원본 보존)
        # (2) self.layernorm2(...):
        #     '더해진 결과'를 '2번 정규화 기계(layernorm2)'로 '정리'합니다.
        # 'out2' (2차 회의 최종본) shape: (B, S_dec, D)
        out2 = self.layernorm2(out1 + attn2)

        # --- 3차 회의 (Feed-Forward Network) + 3차 보고서 취합 (Add & Norm) ---
        # (목적: "초고 문맥(out1)"과 "원본 참조(attn2)"를 '모두 합친(out2)' 정보를 '개인적으로 소화'하기)

        # [7. 개인 학습] 'out2' (2차 회의 최종본)를 'self.ffn' (개인 심화 학습실)에 넣습니다.
        # (내부 동작: 32차원 -> 64차원(확장) -> GELU -> 32차원(압축))
        # 'ffn_output' (개인 학습 결과) shape: (B, S_dec, D)
        ffn_output = self.ffn(out2)

        # [8. 과적합 방지] 개인 학습 결과(ffn_output)에 'self.dropout3'을 적용합니다.
        ffn_output = self.dropout3(ffn_output)

        # [9. 보고서 취합 (Add & Norm)]
        # (1) 'out2 + ffn_output' (Add: Residual Connection):
        #     '개인 학습 전(out2)'과 '개인 학습 후(ffn_output)'를 '더합니다'. (원본 보존)
        # (2) self.layernorm3(...):
        #     '더해진 결과'를 '3번 정규화 기계(layernorm3)'로 '최종 정리'합니다.
        # 'out3' (이 블록의 최종 출력) shape: (B, S_dec, D)
        out3 = self.layernorm3(out2 + ffn_output)

        # 이 '1개 층(블록)'의 모든 분석을 마친 '최종 결과물(out3)'과
        # 2개의 '어텐션 가중치'(w1, w2) (분석/시각화용)를 반환합니다.
        # (이 'out3'는 다음 디코더 층(블록)의 'x' 입력으로 들어가게 됩니다.)
        return out3, attn_weights_block1, attn_weights_block2

# --- 테스트 ---
# '가짜' 디코더 입력 데이터 생성: (B=3, S=10, D=32)
# (이것이 'Outputs (shifted right)'에 해당합니다.)
dec_inp = torch.randn(3, 10, 32)
# (가정: 'enc_out'은 '이전 셀'의 'EncoderLayer' 테스트에서 생성된 [3, 10, 32] 텐서입니다.)
# (enc_out = enc_layer(x=enc_inp, mask=None))

# 'DecoderLayer' 클래스(설계도)로 'dec_layer' (1개 층) 객체를 생성합니다.
# (em_dim=32, num_heads=4, feed_dim=64)
dec_layer = DecoderLayer(32, 4, 64)

# 생성한 'dec_layer' 객체(모델)에 'dec_inp'와 'enc_out' 데이터를 넣고 'forward' 함수를 실행합니다.
# (dec_inp): x (디코더 입력)
# (enc_out): enc_output (인코더 최종본)
# (None, None): 마스크는 사용하지 않고 테스트합니다.
dec_out, w1, w2 = dec_layer(dec_inp, enc_out, None, None)

# 'w2' (attn_weights_block2)의 '모양(shape)'을 출력합니다.
# (w2는 '2차 회의(mha2)'에서 '디코더(Q)'가 '인코더(K)'를 쳐다본 '집중도 행렬'입니다.)
# (예상 모양: (B, H, S_dec, S_enc))
# (B=3, H=4)
# (S_dec = 10, 'q=out1' (즉, dec_inp)의 문장 길이)
# (S_enc = 10, 'k=enc_output'의 문장 길이)
# (최종 예상: [3, 4, 10, 10])
print(w2.shape)

torch.Size([3, 4, 10, 10])


### 포지션 인코딩
멀티헤드 어텐션은 입력 순서를 고려하지 않는(Self-Attention) 구조이므로, 단어의 순서 정보 추가 필요합니다.
이러한 순서정보를 추가하기 위해 토큰 임베딩 결과에 순서정보를 더해줍니다.

포지션 인코딩은 각 포지션의 각도를 구한뒤 sin, cos 함수를 적용하여 상대적인 위치 정보 만들어 냅니다. 이는 병렬처리를 진행하는 트랜스포머에 매우 효율적인 순서 정보를 제공할 수 있습니다.

\begin{aligned}
\text{PE}(pos,\,2i) &= \sin\!\Bigl(\tfrac{pos}{10000^{\,\frac{2i}{d_{\text{model}}}}}\Bigr),\\[8pt]
\text{PE}(pos,\,2i+1) &= \cos\!\Bigl(\tfrac{pos}{10000^{\,\frac{2i}{d_{\text{model}}}}}\Bigr).
\end{aligned}

- 각 토큰의 위치(pos) 에 대해, 특정 각도(주파수)로 변환한 뒤, sin(짝수)과 cos(홀수) 함수를 적용하여 임베딩 차원별로 다른 주기를 갖는 사인파 생성
- 서로 다른 위치(pos) 간의 상대적 거리를 함수의 위상 차이를 통해 모델이 학습



# 포지셔널 인코딩의 변화 흐름
- 초기 트랜스포머 (2017, Vaswani et al.)
    - 사인·코사인 기반 포지셔널 인코딩을 사용.
    - 위치 정보를 수학적으로 고정된 주기 함수로 표현.
- 중간 세대 (BERT, BART, GPT-2 등)
    - **학습형 포지셔널 임베딩(Learned Positional Embedding)**으로 전환.
    - 모델이 직접 위치 벡터를 학습 → 더 유연하고 데이터에 맞게 최적화 가능.
- 최신 세대 (GPT-3.5/4, LLaMA, PaLM, Mistral 등)
    - **RoPE (Rotary Positional Embedding)**을 채택하는 경우가 많음.
    - RoPE는 복소 지수 함수 회전을 이용해 토큰 임베딩을 회전시키며 상대적 위치 정보를 자연스럽게 반영.
    - 긴 문맥에서도 안정적이고, 수학적으로 깔끔하게 확장 가능.




In [ ]:
# (사용자 요청) "Transformer의 “Positional Encoding” 을 시각화"
# (사용자 요청) "문장 내 단어 위치(position)에 따라 사인(sin)과 코사인(cos) 주기로 변하는 값을 만들어, 단어의 위치정보를 벡터로 표현하고 시각화"

# 'matplotlib'의 'pyplot' 모듈을 'plt'라는 별칭으로 가져옵니다.
# (이유: 'pyplot'은 파이썬에서 그래프와 그림(plot)을 그리는 데 사용되는 표준 라이브러리입니다.)
import matplotlib.pyplot as plt
# 'numpy' 라이브러리를 'np'라는 별칭으로 가져옵니다. (수학/배열 연산에 필수)
import numpy as np
# 'torch' 라이브러리를 가져옵니다. (최종 텐서 변환에 필요)
import torch
# (이전 코드에서 사용된 'F' (functional)은 이 코드에서는 필요하지 않습니다.)

# [전체 흐름]
# 트랜스포머는 단어를 '한 번에' 보기 때문에 "I hate you"와 "You hate I"의 '순서'를 모릅니다.
# "Positional Encoding"은 "이 단어는 1번 자리에 있어", "이 단어는 2번 자리에 있어"라는
# '순서(위치) 정보'를 '벡터(숫자 덩어리)'로 만들어서,
# 원래의 '단어 의미 벡터(임베딩)'에 '더해주는' 역할을 합니다.
#
# 이 코드는 그 '위치 정보 벡터'를 생성하고 시각화합니다.

# --- 1. 각도 계산 함수 (핵심 공식) ---
# "Positional Encoding"의 핵심 공식인 '각도(angle)'를 계산하는 함수를 정의합니다.
# pos: (S, 1) 모양의 '단어 위치' 행렬 (예: 0, 1, 2... 49)
# i: (1, D) 모양의 '임베딩 차원' 행렬 (예: 0, 1, 2... 15)
# em_dim: (D) '임베딩 차원'의 총 크기 (스칼라, 예: 16)
def get_angles(pos, i, em_dim):
    # 이것이 "Attention Is All You Need" 논문에 나온 'Positional Encoding'의 핵심 공식입니다.
    # (1) (i // 2): 2i 계산을 위한 정수 나눗셈. (예: [0, 1, 2, 3] -> [0, 0, 1, 1])
    # (2) 2 * (...): 2i 계산. (예: [0, 0, 2, 2])
    # (3) (...) / em_dim: 2i/d_model 계산.
    # (4) 10000 ** (...): 분모 (10000^(2i/d_model)) 계산.
    # (5) pos / (...): pos / (분모) 최종 각도 계산.
    #
    # (이유?) 이 공식은 '위치(pos)'가 멀어질수록 '느리게' 변하는 주기(sin/cos)와,
    # '차원(i)'이 커질수록 '빠르게' 변하는 주기를 '동시에' 생성합니다.
    # (결과) 각 위치(단어)마다 '고유한' 사인/코사인 패턴을 가진 벡터가 생성됩니다.
    # (반환값 shape: (S, D) -> (50, 16))

    # [공식의 직관적 의미: 시계 바늘 원리]
    # 이 공식은 차원(i)이 커질수록, 숫자가 변하는 속도(주파수)를 점점 느리게 만듭니다.
    #
    # [앞쪽 차원 vs 뒤쪽 차원]
    # 단어 하나는 여러 개의 숫자 리스트(예: 512개)로 표현됩니다.
    # 이때 리스트의 '순서(인덱스 i)'가 바로 '차원'입니다.
    #
    # 1. 앞쪽 차원 (i가 0에 가까울 때):
    #    - 분모가 작음 -> 결과값이 큼 -> 숫자가 '초침'처럼 빠르게 뱅글뱅글 변함.
    #
    # 2. 뒤쪽 차원 (i가 em_dim에 가까울 때):
    #    - 분모가 큼 -> 결과값이 작음 -> 숫자가 '시침'처럼 아주 천천히 변함.
    #
    # [결과] "빠른 바늘"과 "느린 바늘"을 조합해서 각 위치(pos)만의 고유한 '시간(지문)'을 만듭니다.
    #
    # - i가 0일 때 (앞쪽 차원): 분모가 1에 가까움 -> '초침'처럼 미친듯이 빠르게 숫자가 변함.
    # - i가 클 때 (뒤쪽 차원): 분모가 10000에 가까움 -> '시침'처럼 아주 천천히 숫자가 변함.
    #
    # 이렇게 차원마다 속도를 다르게 섞으면, 모든 위치(pos)가 겹치지 않는 '고유한 패턴'을 갖게 됩니다.

    # 초침(앞쪽 차원)만 있다면?
    # 초침은 60초마다 한 바퀴를 돕니다.
    # 지금이 '1분 5초'인지, '2분 5초'인지 초침만 보고는 구별할 수 없습니다. (5초 위치에서 겹침)
    # 초침 + 분침 + 시침이 다 있다면?
    # 초침은 빨리 돌고, 분침은 중간, 시침은 느리게 돕니다.
    # 이 '서로 다른 속도의 바늘들'의 위치를 조합하면, "12시 30분 15초"라는 **유일한 시간(위치)**을 정확히 찍어낼 수 있습니다.

    # [수식 분해]
    # 1. (i // 2): 짝수(0)와 홀수(1) 인덱스를 한 쌍으로 묶습니다. (sin, cos 짝꿍용)
    # 2. 10000 ** (...): '속도 조절기'입니다. 차원(i)이 커질수록 분모를 10000배까지 키웁니다.
    # 3. pos / (...): 현재 내 위치(pos)가 시계의 어디쯤인지 각도를 계산합니다.

    return pos / (10000 ** ((2 * (i // 2)) / em_dim))

# --- 2. 위치 인코딩 행렬 생성 함수 ---
# 'get_angles' 함수를 사용해 실제 '위치 인코딩 행렬'을 생성하는 함수를 정의합니다.
# position: (S) 문장의 최대 길이 (스칼라, 예: 50)
# em_dim: (D) 임베딩 차원의 총 크기 (스칼라, 예: 16)
def positional_encoding(position, em_dim):
    # 'get_angles' 함수를 호출하여 '각도 행렬'을 계산합니다.
    angle_rads = get_angles(
        # 'pos' 인자 (위치):
        # (1) np.arange(position): 'pos' (위치)에 해당하는 1D 배열을 생성합니다. (예: [0, 1, ..., 49])
        # (2) [:, np.newaxis]: 1D 배열(S,)을 2D '열 벡터'(S, 1)로 변환합니다. (모양: [50, 1])
        # (이유?) (S, 1)과 (1, D)를 '브로드캐스팅' 연산하여 (S, D) 행렬을 만들기 위함입니다.
        np.arange(position)[:, np.newaxis],

        # 'i' 인자 (차원):
        # (1) np.arange(em_dim): 'i' (차원)에 해당하는 1D 배열을 생성합니다. (예: [0, 1, ..., 15])
        # (2) [np.newaxis, :]: 1D 배열(D,)을 2D '행 벡터'(1, D)로 변환합니다. (모양: [1, 16])
        np.arange(em_dim)[np.newaxis, :],

        # 'em_dim' 인자 (차원 크기):
        em_dim
    )
    # (최종 결과) 'get_angles'는 (50, 1)과 (1, 16)을 입력받아 (50, 16) 모양의 '각도 행렬' (angle_rads)을 반환합니다.

    # 짝수 인덱스에는 sin, 홀수 인덱스에는 cos

    # 'angle_rads' (50, 16) 행렬의 '모든 행'(:)에 대해,
    # '0::2' (0번, 2번, 4번... '짝수 인덱스' 열)을 선택합니다.
    # 그 위치의 '각도' 값들에 'np.sin' (사인) 함수를 적용하여 값을 '덮어씁니다'.
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # 'angle_rads' (50, 16) 행렬의 '모든 행'(:)에 대해,
    # '1::2' (1번, 3번, 5번... '홀수 인덱스' 열)을 선택합니다.
    # 그 위치의 '각도' 값들에 'np.cos' (코사인) 함수를 적용하여 값을 '덮어씁니다'.
    # (이유?) 짝/홀수 인덱스에 sin/cos를 교차 적용하는 것이 논문에서 제안한 방식입니다.
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    # '...' (ellipsis)는 "나머지 모든 차원" (즉, 50, 16)을 의미합니다.
    # 'np.newaxis' (맨 앞): 텐서의 '0번 차원'에 '배치(Batch)' 차원(크기 1)을 '추가'합니다.
    # (모양 변화) (50, 16) -> (1, 50, 16) (B, S, D)
    # (이유?) 파이토치 모델은 (B, S, D) 형태의 3D 텐서를 기본 입력으로 사용하기 때문입니다.
    pos_encoding = angle_rads[np.newaxis, ...]  # 배치 축 추가 (1, position, em_dim)

    # 'np.array' (Numpy 배열) 형태인 'pos_encoding'을
    # 'torch.tensor' (파이토치 텐서)로 '변환'하여 반환합니다.
    # (dtype=torch.float32): 딥러닝 연산에 표준인 '32비트 부동소수점' 타입으로 지정합니다.
    return torch.tensor(pos_encoding, dtype=torch.float32)

# --- 3. 실행 및 시각화 ---

# (테스트 설정 1) 문장의 최대 길이(S)를 50 (단어 50개)으로 설정합니다.
position = 50
# (테스트 설정 2) 임베딩 차원(D)을 16 (16차원 벡터)으로 설정합니다. (시각화를 위해 작게 설정)
em_dim = 16

# 위에서 정의한 'positional_encoding' 함수를 (50, 16) 설정으로 실행합니다.
# 'pos_encoding' 변수에는 (1, 50, 16) 모양의 텐서가 저장됩니다.
pos_encoding = positional_encoding(position, em_dim)
# 생성된 'pos_encoding' 텐서의 '모양(shape)'을 출력합니다.
print(pos_encoding.shape) # (예상 출력: torch.Size([1, 50, 16]))

# -------------------------------------------------------------------------
# 직관적 설명:
# - 출력은 (1, 50, 16) → 배치 1개, 50개 위치, 16차원 벡터
# - 각 위치마다 16차원의 sin/cos 값이 들어 있음
# - 차원이 커질수록 파형이 느려지고, 짝/홀 차원은 위상이 달라짐
# - 결과적으로 "위치마다 고유한 주기적 벡터 서명"이 생성됨
# - 이 벡터를 단어 임베딩에 더하면 모델은 단어 의미 + 위치 정보를 동시에 사용 가능
# -------------------------------------------------------------------------

# --- (시각화 코드 시작) ---
# 'matplotlib'을 사용하여 생성된 위치 인코딩을 '시각화'합니다.
# (이 부분이 사용자 요청 설명의 "시각화"에 해당합니다.)

# 그래프를 그릴 '도화지(figure)'를 준비하고, 크기를 (가로 10인치, 세로 6인치)로 설정합니다.
plt.figure(figsize=(10, 6))

# 'pos_encoding' 텐서는 (B, S, D) (1, 50, 16) 모양입니다.
# '[0]' 인덱싱으로 '배치' 차원을 제거하여 (S, D) (50, 16) 모양의 텐서를 가져옵니다.
# '.numpy()': 파이토치 텐서를 다시 'Numpy 배열'로 변환합니다 (matplotlib은 Numpy 배열을 입력받음).
# 'plt.pcolormesh': 2D 행렬(50x16)의 값을 '색상 지도(heatmap)'로 그려주는 함수입니다.
# (가로축(x): 16개 차원, 세로축(y): 50개 위치)
# 'cmap='viridis'': 색상 테마를 'viridis' (노랑-초록-파랑)로 설정합니다.
plt.pcolormesh(pos_encoding[0].numpy(), cmap='viridis')

# x축에 'Embedding Dimension (d_model)'이라는 라벨을 붙입니다.
plt.xlabel('Embedding Dimension (d_model)')
# y축에 'Position'이라는 라벨을 붙입니다.
plt.ylabel('Position')

# 오른쪽에 '색상 막대(colorbar)'를 추가하여, 색상이 어떤 값(-1 ~ +1)에 해당하는지 보여줍니다.
plt.colorbar()
# 그래프 전체의 '제목'을 설정합니다.
plt.title('Positional Encoding Visualization')

# 최종적으로 '준비된 그래프를 화면에 표시'합니다.
plt.show()

torch.Size([1, 50, 16])


### 최종 인코더와 디코더

지금까지 구현한 인코더 레이어와 디코더 레이어, 임베딩 레이어, 포지션 인코딩을 결합하여 Transformer 아키텍쳐를 만듭니다.

<center><img src="https://arxiv.org/html/1706.03762v7/extracted/1706.03762v7/Figures/ModalNet-21.png" width="300"/></center>

In [ ]:
# (이 코드를 실행하려면 'torch', 'nn', 'math'가 import 되어 있어야 하며,
# 'EncoderLayer', 'MultiHeadAttention', 'point_wise_feed_forward_network',
# 'positional_encoding'이 이전 코드에서 정의되어 있어야 합니다.)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# (이전 코드... EncoderLayer, MultiHeadAttention 등이 여기에 있다고 가정)
# ...
# def positional_encoding(position, em_dim): ...
# class EncoderLayer(nn.Module): ...
# ...

# [전체 흐름]
# 이 코드는 'Encoder' (인코더) '전체'를 구현한 것입니다.
# "원본 문서 분석팀" 비유에서, 이 클래스는 "분석팀 공장" 그 자체입니다.
#
# 이 공장은 'Nx' (예: 6개)의 'EncoderLayer' (분석팀 블록/1학년~6학년)를
# '차곡차곡 쌓아서(Stacking)' 완성된 "인코더"를 만듭니다.
#
# [공장의 작업 순서 (요약)]
# 1. (__init__): '임베딩' 부품, '포지셔널 인코딩' 부품, '6개의 EncoderLayer' 부품을 '미리' 만들어 둡니다.
# 2. (forward):
#    (1) '문장(x)'을 '임베딩(벡터화)'합니다.
#    (2) '포지셔널 인코딩(위치정보)'을 '더합니다'.
#    (3) '드롭아웃'을 적용합니다.
#    (4) 이 결과를 '1층(EncoderLayer[0])'에 넣습니다.
#    (5) 1층의 출력을 '2층(EncoderLayer[1])'에 넣습니다.
#    (6) ...
#    (7) 'N층(EncoderLayer[N-1])'의 '최종 출력'을 반환합니다.
class Encoder(nn.Module):
    '''
    # (클래스 설명 주석: 이 클래스가 무엇을 하는지 요약)
    #   1) Embedding: 단어 ID → 임베딩 벡터로 변환 후 스케일링
    #   2) Positional Encoding: 위치 정보(sin/cos)를 임베딩에 더함
    #   3) Dropout: 입력 임베딩+포지션에 드롭아웃 적용
    #   4) Stacked EncoderLayers (각 층마다): (N번 반복)
    #     - Multi-Head Self-Attention
    #     - Dropout + Residual + LayerNorm
    #     - Feed Forward Network (FFN)
    #     - Dropout + Residual + LayerNorm
    '''

    # [지금 하는 일: '인코더 공장' 초기화 함수(생성자) 정의]
    # 'Encoder'라는 '설계도'로 '객체(모델)'를 만들 때
    # 이 __init__ 함수가 호출되어, 인코더 '전체'에 필요한 '부품'들을 '미리' 준비합니다.
    # num_layers: (예: 6) 'EncoderLayer'를 '몇 층' 쌓을지 (Nx)
    # em_dim: (예: 512) 모델의 기본 '작업 차원'
    # num_heads: (예: 8) Multi-Head Attention에서 사용할 '전문가(헤드)'의 수
    # feed_dim: (예: 2048) FFN의 '중간 확장' 차원
    # input_vocab_size: (예: 30000) '입력' 단어장의 총 단어 개수
    # maximum_position_encoding: (예: 10000) 이 모델이 처리할 '최대' 문장 길이
    # rate: (예: 0.1) 드롭아웃 비율
    def __init__(self, num_layers, em_dim, num_heads, feed_dim,
                 input_vocab_size, maximum_position_encoding, rate=0.1):
        # 부모 클래스(nn.Module)의 초기화 함수를 호출합니다. (파이토치 모델 정의 시 필수)
        super(Encoder, self).__init__()
        # 512 (작업 차원)를 self.em_dim 변수에 저장합니다.
        self.em_dim = em_dim
        # 6 (층 수)를 self.num_layers 변수에 저장합니다.
        self.num_layers = num_layers

        # --- 부품 1: 임베딩 레이어 (단어 ID -> 벡터) ---
        # 'nn.Embedding' (조회 테이블) 부품을 생성합니다.
        # (30000개 단어를 512차원 벡터로 변환할 수 있도록 준비)
        self.embedding = nn.Embedding(input_vocab_size, em_dim)

        # --- 부품 2: 포지셔널 인코딩 (위치 정보) ---
        # 'positional_encoding' (이전 코드에서 정의한 함수)를 '미리' 호출합니다.
        # (maximum_position_encoding=10000, em_dim=512)
        # (결과) 'self.pos_encoding'에는 (1, 10000, 512) 모양의
        # '위치 정보 행렬(sin/cos)'이 '미리 계산되어' 저장됩니다.
        # (이유? 이 값은 '학습'되는 값이 아니라 '고정된' 값이므로,
        # 훈련 시작 전에 '딱 한 번'만 만들어두고 '재사용'하는 것이 효율적입니다.)
        self.pos_encoding = positional_encoding(maximum_position_encoding, em_dim)

        # --- 부품 3: N개의 'EncoderLayer' (분석팀 1~N층) ---
        # 'nn.ModuleList'는 '파이토치 레이어(Module)'를 담을 수 있는 '파이썬 리스트'입니다.
        # (이유? 그냥 파이썬 리스트 [ ]에 담으면, 파이토치가 '학습할 파라미터'로 '인식'하지 못합니다.
        # 'nn.ModuleList'에 담아야 "아, 이것들도 내가 학습시킬 부품들이구나"라고 '등록'이 됩니다.)
        self.enc_layers = nn.ModuleList([
            # 'num_layers' (예: 6)번 만큼 'for' 루프를 돕니다.
            # (루프 1) EncoderLayer(512, 8, 2048, 0.1) 객체 '1개' (1층)를 만듭니다.
            # (루프 2) EncoderLayer(512, 8, 2048, 0.1) 객체 '1개' (2층)를 만듭니다.
            # (중요!) 1층과 2층은 '서로 다른' 가중치를 가진 '별개의' 객체입니다.
            # ( ... 6층까지 반복 ... )
            EncoderLayer(em_dim, num_heads, feed_dim, rate)
            for _ in range(num_layers)
        ])

        # --- 부품 4: 드롭아웃 ---
        # '임베딩 + 포지션'을 '더한 직후'에 '딱 한 번' 사용할 '공용' 드롭아웃 레이어입니다.
        self.dropout = nn.Dropout(rate)

    # [지금 하는 일: '인코더 공장'의 실제 동작(순전파) 흐름 정의]
    # '데이터가 입력되었을 때(forward)' '어떤 순서로 작동'할지 '흐름'을 정의합니다.
    # x: (B, S) (예: [32, 20]) - '단어 ID' 텐서 (디지털화 전 '원본 문서')
    # mask: (B, 1, 1, S) (예: [32, 1, 1, 20]) - '패딩 마스크' (0([PAD])인 부분을 무시하기 위함)
    def forward(self, x, mask=None):
        # 'x' (B, S)에서 'S' (문장 길이, 예: 20)를 가져옵니다.
        seq_len = x.size(1)

        # --- (1) 임베딩 ---
        # 'self.embedding' (부품 1)을 통과시킵니다.
        # (B, S) -> (B, S, D)
        # (예: [32, 20] -> [32, 20, 512])
        x = self.embedding(x)

        # --- (2) 스케일링 (Scaling) ---
        # [목적: 단어 의미(Embedding)의 '체급' 키우기]
        #
        # (이유?)
        # 1. 위치 정보(Positional Encoding)는 sin/cos 값이라 -1 ~ 1 사이의 큰 크기를 가집니다. (목소리가 큼)
        # 2. 반면, 임베딩(x)은 초기화될 때 0.01 처럼 아주 작은 소수점 값을 가집니다. (목소리가 작음)
        #
        # (문제점)
        # 이 둘을 그냥 더하면(x + pe), 값이 큰 '위치 정보'에 '단어 의미'가 완전히 묻혀버립니다.
        # (예: 0.01(의미) + 1.0(위치) = 1.01 -> 의미가 반영됐는지 티가 안 남)
        #
        # (해결책)
        # 단어 벡터(x)에 sqrt(d_model) (예: sqrt(512) ≈ 22.6배)을 곱해서 숫자를 '뻥튀기' 해줍니다.
        # 이렇게 해야 단어 의미와 위치 정보가 대등한 크기로 잘 섞이게 됩니다.
        x = x * math.sqrt(self.em_dim)

        # --- (3) 포지셔널 인코딩 '더하기' ---
        # 'self.pos_encoding' (미리 만든 '부품 2') (1, 10000, 512)에서
        # '[:, :seq_len, :]' (슬라이싱): '딱 지금 필요한 길이(S=20)'만큼 '잘라냅니다'.
        # (결과) pe shape: (1, 20, 512)
        # '.to(x.device)': 만약 모델이 GPU에 있다면, 'pe' 조각도 GPU로 보냅니다. (필수!)
        pe = self.pos_encoding[:, :seq_len, :].to(x.device)

        # 'x' (의미 벡터)와 'pe' (위치 벡터)를 '더합니다'.
        # (B, S, D) + (1, S, D) -> (B, S, D)
        # (pe(1,S,D)가 'B'(32)개 문장에 '자동으로 복사'(브로드캐스팅)되어 더해집니다.)
        # (결과) 'x'는 이제 '의미'와 '위치' 정보를 '모두' 가진 벡터가 되었습니다.
        x = x + pe

        # 'self.dropout' (부품 4)를 '합쳐진 결과'에 '한 번' 적용합니다.
        x = self.dropout(x)

        # --- (4) N개의 'EncoderLayer' (부품 3) 통과 ---
        # 'self.num_layers' (예: 6)번 만큼 'for' 루프를 돕니다.
        for i in range(self.num_layers):
            # (루프 i=0): 'x'(최초 입력)를 'self.enc_layers[0]' (1층)에 넣습니다.
            # (mask): '패딩 마스크'는 '모든' 층(Layer)의 'MHA'에서 필요하므로 '계속 전달'합니다.
            # 'x'가 '1층 통과 결과'로 '업데이트'됩니다. (모양은 [32, 20, 512]로 동일)
            #
            # (루프 i=1): '업데이트된 x'(1층 결과)를 'self.enc_layers[1]' (2층)에 넣습니다.
            # 'x'가 '2층 통과 결과'로 '업데이트'됩니다.
            #
            # (... 6층까지 반복 ...)
            x = self.enc_layers[i](x, mask)

        # '6층'까지 모두 통과한 '최종 분석본(x)' (B, S, D)을 반환합니다.
        # (이것이 '디코더'로 전달될 'K'와 'V'가 됩니다.)
        return x

In [ ]:
# (이 코드를 실행하려면 'torch', 'nn', 'math'가 import 되어 있어야 하며,
# 'DecoderLayer', 'positional_encoding'이 이전 코드에서 정의되어 있어야 합니다.)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# (이전 코드... DecoderLayer, positional_encoding 등이 여기에 있다고 가정)
# ...

# [전체 흐름]
# 이 코드는 'Decoder' (디코더) '전체'를 구현한 것입니다.
# "단어 생성 작가팀" 비유에서, 이 클래스는 "작가팀 공장" 그 자체입니다.
#
# 이 공장은 'Nx' (예: 6개)의 'DecoderLayer' (작가팀 블록/1학년~6학년)를
# '차곡차곡 쌓아서(Stacking)' 완성된 "디코더"를 만듭니다.
#
# [공장의 작업 순서 (요약)]
# 1. (__init__): '임베딩' 부품, '포지셔널 인코딩' 부품, '6개의 DecoderLayer' 부품을 '미리' 만들어 둡니다.
# 2. (forward):
#    (1) '초고(x)'를 '임베딩(벡터화)'하고 '순서(포지션)'를 더합니다.
#    (2) 이 재료를 '1층(DecoderLayer[0])'에 넣습니다.
#        (이때, '인코더의 최종 분석본(enc_output)'도 같이 넣어줍니다.)
#    (3) 1층의 출력을 '2층(DecoderLayer[1])'에 넣습니다.
#    (4) ...
#    (5) 'N층'까지 통과한 '최종 출력'과 '어텐션 가중치(참조 기록)'를 반환합니다.
class Decoder(nn.Module):
    '''
    # (클래스 설명 주석)
    #   1) Embedding + 스케일링: 디코더 입력 토큰을 벡터로 변환하고 크기를 보정
    #   2) Positional Encoding: 위치 정보를 sin/cos 패턴으로 임베딩에 더함
    #   3) Dropout: 임베딩+포지션에 드롭아웃 적용 (과적합 방지)
    #   4) Stacked DecoderLayers (각 층마다):
    #     - Masked Multi-Head Self-Attention: 미래 토큰을 가린 자기 어텐션
    #     - Encoder–Decoder Attention: 인코더 출력과의 관계 학습
    #     - Feed Forward Network (FFN): 각 단어 벡터 비선형 변환
    #     - 각 블록마다 Dropout + Residual + LayerNorm으로 안정화
    #     - Attention Weights 저장: block1(자기), block2(인코더-디코더) 기록
    '''

    # [지금 하는 일: '디코더 공장' 초기화 함수(생성자) 정의]
    # 'Decoder' 객체를 만들 때 필요한 모든 부품을 미리 준비합니다.
    # num_layers: (예: 6) 'DecoderLayer'를 몇 층 쌓을지
    # em_dim: (예: 512) 모델의 기본 작업 차원
    # num_heads: (예: 8) 전문가(헤드) 수
    # feed_dim: (예: 2048) FFN 확장 차원
    # target_vocab_size: (예: 8000) '출력' 단어장의 총 단어 개수 (한글 단어 개수)
    # maximum_position_encoding: (예: 10000) 최대 문장 길이
    def __init__(self, num_layers, em_dim, num_heads, feed_dim,
                 target_vocab_size, maximum_position_encoding, rate=0.1):
        # 부모 클래스(nn.Module) 초기화 (필수)
        super(Decoder, self).__init__()
        # 차원(512)과 층수(6)를 저장합니다.
        self.em_dim = em_dim
        self.num_layers = num_layers

        # --- 부품 1: 임베딩 레이어 ---
        # 'nn.Embedding': 출력 단어 ID(예: "나는")를 벡터로 변환하는 부품입니다.
        # (8000개 단어를 512차원 벡터로 변환)
        # [8000과 '나는'의 관계]
        # - 8000 (target_vocab_size): 우리 가게가 파는 '전체 메뉴 개수'입니다.
        # - "나는": 그 8000개 메뉴 중 '하나'입니다. (예: 254번 메뉴)
        #
        # 이 레이어는 8000개의 단어 각각에 대해 512차원 벡터(레시피)를 미리 준비해 둡니다.
        # 나중에 "254번(나는) 주세요" 하면, 준비된 254번 벡터를 꺼내줍니다.
        self.embedding = nn.Embedding(target_vocab_size, em_dim)

        # --- 부품 2: 포지셔널 인코딩 ---
        # 'positional_encoding': 위치 정보 행렬을 미리 계산해 둡니다.
        self.pos_encoding = positional_encoding(maximum_position_encoding, em_dim)

        # --- 부품 3: N개의 'DecoderLayer' (작가팀 1~N층) ---
        # 'nn.ModuleList'에 6개의 DecoderLayer를 담습니다.
        self.dec_layers = nn.ModuleList([
            # (중요!) 인코더와 마찬가지로, 각 층은 서로 다른 가중치를 가진 '별개의' 객체입니다.
            # 1학년 작가팀, 2학년 작가팀... 6학년 작가팀을 고용합니다.
            DecoderLayer(em_dim, num_heads, feed_dim, rate)
            for _ in range(num_layers)
        ])

        # --- 부품 4: 드롭아웃 ---
        self.dropout = nn.Dropout(rate)

    # [지금 하는 일: '디코더 공장'의 실제 동작(순전파) 흐름 정의]
    # x: (B, S_dec) - 'Outputs (shifted right)' (지금까지 쓴 초고, 예: [BOS], "나는")
    # enc_output: (B, S_enc, D) - '인코더'의 최종 분석본 (도서관/원본 책 내용)
    # look_ahead_mask: (S_dec, S_dec) - '미래 훔쳐보기 방지' 마스크 (필수!)
    # padding_mask: (B, 1, 1, S_enc) - 인코더의 [PAD] 부분을 보지 않게 하는 마스크
    def forward(self, x, enc_output, look_ahead_mask=None, padding_mask=None):
        # 입력된 문장의 길이(S_dec)를 잽니다.
        seq_len = x.size(1)

        # 'attention_weights': 각 층에서 "어디를 쳐다봤는지" 기록할 빈 사전을 만듭니다.
        # (나중에 시각화해서 분석할 때 매우 중요합니다.)
        attention_weights = {}

        # --- (1) 재료 손질: 임베딩 및 포지션 인코딩 ---

        # 1-1. 임베딩: 단어 ID(x)를 벡터로 변환합니다. (B, S) -> (B, S, D)
        x = self.embedding(x)
        # 1-2. 스케일링: 값의 크기를 키워줍니다. (논문 공식)
        x = x * math.sqrt(self.em_dim)

        # 1-3. 포지셔널 인코딩 더하기:
        # 미리 만들어둔 '위치 행렬'에서 현재 문장 길이(seq_len)만큼 잘라서 가져옵니다.
        pe = self.pos_encoding[:, :seq_len, :].to(x.device)
        # '의미(x)' + '순서(pe)'를 합칩니다.
        x = x + pe

        # 1-4. 드롭아웃 적용
        x = self.dropout(x)

        # --- (2) N개의 'DecoderLayer' (작가팀) 통과 ---

        # 'self.num_layers' (예: 6)번 만큼 루프를 돕니다.
        for i in range(self.num_layers):
            # [중요] 각 층(dec_layers[i])에 다음 4가지를 모두 넣어줍니다.
            # 1. x : 현재까지 처리된 '초고' (이전 층의 출력)
            # 2. enc_output : '인코더'의 최종 분석본 (도서관). *모든 층이 똑같은 걸 봅니다.*
            # 3. look_ahead_mask : 컨닝 방지 마스크
            # 4. padding_mask : 인코더 패딩 무시 마스크
            x, block1, block2 = self.dec_layers[i](x, enc_output,
                                                   look_ahead_mask,
                                                   padding_mask)

            # [기록] 이번 층(i)에서 계산된 '어텐션 가중치'들을 저장합니다.
            # block1: 1차 회의(Self-Attention) 가중치 ("초고의 어디를 봤나?")
            # block2: 2차 회의(Cross-Attention) 가중치 ("원본의 어디를 봤나?")
            # (이걸 저장해두면 나중에 "아, 모델이 '그것'을 번역할 때 'it'을 쳐다봤구나"라고 알 수 있습니다.)
            attention_weights[f'decoder_layer{i+1}_block1'] = block1
            attention_weights[f'decoder_layer{i+1}_block2'] = block2

        # 모든 층(6층)을 통과한 '최종 결과물(x)'과 '시선 기록(weights)'을 반환합니다.
        # (x의 모양: [B, S_dec, D]) -> 이제 이것이 'Linear' 층으로 가서 단어 점수표가 됩니다.
        return x, attention_weights

In [ ]:
# [전체 흐름: 트랜스포머 전체 파이프라인 테스트]
# 이제까지 우리가 열심히 만든 부품들(Encoder, Decoder, Mask 등)을 모두 조립해서,
# 데이터가 입력되고 -> 인코더를 거쳐 -> 디코더를 나오는 '전체 과정'을 확인하는 코드입니다.

# 1. 입력 데이터 준비 (가짜 데이터 생성)
# eco_inp: "인코더 입력" (원본 문장). [배치1, 길이10].
# (1, 2, 3, 4, 5, 6은 단어이고, 뒤에 0, 0, 0, 0은 패딩입니다.)
eco_inp = torch.Tensor([[1, 2, 3, 4, 5, 6, 0, 0, 0, 0]]).long()

# dco_inp: "디코더 입력" (Outputs shifted right). [배치1, 길이10].
# (교사 강요를 위한 힌트입니다. 정답지보다 한 칸씩 밀려 있습니다.)
dco_inp = torch.Tensor([[1, 2, 3, 4, 5, 6, 7, 8, 0, 0]]).long()

# 2. 마스크 준비 (가리개 생성)
# pad_mask: "인코더용 패딩 마스크".
# (인코더가 분석할 때, eco_inp의 뒤쪽 '0' 4개를 "쳐다보지 말라"고 알려줍니다.)
pad_mask = create_padding_mask(eco_inp)

# lh_mask: "디코더용 룩어헤드(미래 방지) 마스크".
# (디코더가 글을 쓸 때, 자기보다 미래에 있는 단어를 "컨닝하지 말라"고 가려줍니다.)
lh_mask = create_look_ahead_mask(10)

# 3. 인코더(Encoder) 생성 및 실행
# [설정] 2층(layers), 32차원(em_dim), 4명 전문가(heads), 64차원 확장(feed_dim), 단어장 10개, 최대길이 100
ecoder = Encoder(2, 32, 4, 64, 10, 100)

# [실행] "분석팀(Encoder)"이 "원본 문장(eco_inp)"을 "패딩 무시(pad_mask)"하며 분석합니다.
# (결과) eco_out: '최종 분석본(도서관)'. (K, V로 사용됨)
eco_out = ecoder(eco_inp, pad_mask)

# [결과 확인] shape: [1, 10, 32]
# 1: 배치 크기 (문장 1개)
# 10: 문장 길이 (단어 10개)
# 32: 임베딩 차원 (각 단어가 32개의 숫자로 표현된 '깊은 의미'를 가짐)
print(f'ecoder out:{eco_out.shape}')

# 4. 디코더(Decoder) 생성 및 실행
# [설정] 인코더와 동일한 크기(32차원)로 설정해야 대화가 통합니다.
decoder = Decoder(2, 32, 4, 64, 10, 100)

# [실행] "작가팀(Decoder)"이 집필을 시작합니다.
# 입력 1 (dco_inp): "지금까지 쓴 초고" (힌트)
# 입력 2 (eco_out): "인코더가 만든 분석본" (도서관 참조 자료) -> K, V
# 입력 3 (lh_mask): "컨닝 방지 마스크" (1차 회의용)
# 입력 4 (pad_mask): "인코더 패딩 무시 마스크" (2차 회의용 - 도서관의 빈 책장은 보지 마라)
dec_out, attn = decoder(dco_inp, eco_out, lh_mask, pad_mask)

# [결과 확인] shape: [1, 10, 32]
# (주의!) 아직 '단어 점수(vocab size)'로 변환되기 직전의 '최종 생각 벡터'입니다.
# 1: 배치 크기
# 10: 문장 길이 (각 위치마다 예측한 결과가 나옴)
# 32: 임베딩 차원 (각 위치의 '최종 생각'이 32차원 벡터로 표현됨)
# (이후에 Linear(32 -> 단어장크기) 레이어를 통과해야 최종 확률이 나옵니다.)
print(f'decoder out:{dec_out.shape}')

ecoder out:torch.Size([1, 10, 32])
decoder out:torch.Size([1, 10, 32])


In [ ]:
# [결과 분석: 트랜스포머의 뇌구조 들여다보기]
# 디코더의 2번째 블록(Layer 2)이 계산한 '어텐션 가중치'를 출력합니다.
# (attn 딕셔너리에는 각 층의 어텐션 점수들이 저장되어 있습니다.)

print('--- [1번 어텐션] Masked Self-Attention (초고 검토) ---')
# 상황: 디코더가 "지금까지 쓴 글(dco_inp)"을 스스로 훑어보는 중입니다.
# 규칙: "미래의 단어는 절대 훔쳐보지 마라!" (Look-Ahead Mask 적용)
print(attn['decoder_layer2_block1'][0])

# [▼▼▼ 출력된 텐서(숫자) 해석 ▼▼▼]
# 행(Row): 현재 시점의 단어 / 열(Col): 바라보는 과거의 단어
#
# tensor([[1.0000, 0.0000, 0.0000, ...],  <-- 1번째 단어는 '자기 자신(1번)'만 봅니다. (나머지 0)
#         [0.5407, 0.4593, 0.0000, ...],  <-- 2번째 단어는 '1번, 2번'만 봅니다. (3번부터 0)
#         [0.2190, 0.5547, 0.2263, 0.0000, ...], <-- 3번째 단어는 '1, 2, 3번'만 봅니다.
#         ... ])
#
# [핵심 확인 포인트]
# 오른쪽 위 대각선 부분이 전부 '0.0000' 인 것을 확인하세요! (삼각형 모양)
# -> 이것이 바로 "미래 단어를 컨닝하지 못하게 가렸다"는 증거입니다.


print('\n' + '='*50 + '\n')


print('--- [2번 어텐션] Encoder-Decoder Attention (원본 참조) ---')
# 상황: 디코더가 "인코더의 결과물(eco_out)"을 참고하는 중입니다.
# 규칙: "원본 문장의 빈칸(PAD)은 절대 보지 마라!" (Padding Mask 적용)
print(attn['decoder_layer2_block2'][0])

# [▼▼▼ 출력된 텐서(숫자) 해석 ▼▼▼]
# 행(Row): 디코더가 생성 중인 단어 / 열(Col): 인코더의 원본 단어들
#
# tensor([[0.2106, ..., 0.1416, 0.0000, 0.0000, 0.0000, 0.0000], <-- 첫 줄
#         [0.1008, ..., 0.1302, 0.0000, 0.0000, 0.0000, 0.0000], <-- 둘째 줄
#         ... ])
#                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#                               여기를 보세요! (오른쪽 끝 4개)
#
# [핵심 확인 포인트]
# 모든 줄의 오른쪽 끝 4개 숫자가 전부 '0.0000' 입니다.
# -> 아까 우리가 eco_inp를 만들 때 [[1, 2, 3, 4, 5, 6, 0, 0, 0, 0]] 이렇게 뒤에 0을 4개 넣었죠?
# -> 모델이 그 0(PAD)이 있는 위치는 "정보가 없다"고 판단하고 무시했다는 증거입니다.

1번 어텐션
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000],
         [0.5407, 0.4593, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000],
         [0.2190, 0.5547, 0.2263, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000],
         [0.1607, 0.1606, 0.2533, 0.4254, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000],
         [0.1312, 0.2599, 0.1176, 0.2697, 0.2217, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000],
         [0.1131, 0.1660, 0.1661, 0.2487, 0.1579, 0.1483, 0.0000, 0.0000,
          0.0000, 0.0000],
         [0.0879, 0.1325, 0.1657, 0.2488, 0.1295, 0.0890, 0.1466, 0.0000,
          0.0000, 0.0000],
         [0.0638, 0.0764, 0.1059, 0.2394, 0.0804, 0.1274, 0.1662, 0.1406,
          0.0000, 0.0000],
         [0.0585, 0.1082, 0.1127, 0.2024, 0.1059, 0.0530, 0.0726, 0.1733,
          0.1134, 0.0000],
         [0.0548, 0.0936, 0.1068, 0.1837, 0.0943, 0.0486, 0.0742, 0.1452,
          

### Transformer 모델링

인코더와 디코더를 결합하고 최종 분류기를 더한 Transformer 구조를 만듭니다. 이때 인코더와 디코더에 넣어줄 마스크를 `forward` 단계에서 생성합니다.

완성된 Transformer모델의 `forward`는 학습을 위한 순전파로 타겟이 같이 입력 되어야 합니다. 따라서 학습된 Transformer모델로 텍스트를 생성하는 작업은 `forward` 함수를 활용하기에 적절 하지 않습니다.
해당 예시에선느 학습된 Transformer를 활용해 텍스트를 생성하기 위한 `generate` 함수를 따로 구현 합니다.

**generate 함수**
- 인코더 입력될 토큰셋과 디코더에 입력될 시작 토큰 하나만 입력으로 받아 자기회귀(Autoregressive)방식으로 토큰을 생성
- 최대 토큰 길이를 설정하여 디코더의 토큰 예측을 반복하고 문장 끝 토큰이 나오는 경우 반복을 중단
- 토큰 id를 실제 단어 토큰으로 변환하여 출력



In [ ]:
import torch
import torch.nn as nn

# [전체 흐름: 트랜스포머 공장 완공]
# 인코더(분석팀)와 디코더(작가팀)를 고용하고,
# 이들을 관리하며 실제 '훈련'과 '생성'을 지휘하는 총괄 책임자 클래스입니다.
class Transformer(nn.Module):
    def __init__(self, num_layers, em_dim, num_heads, feed_dim,
                 input_vocab_size, target_vocab_size,
                 pe_input, pe_target, rate=0.1):
        super(Transformer, self).__init__()

        # --- 1. 분석팀(Encoder) 고용 ---
        # 입력 문장(소스)을 이해하고 압축하는 역할
        self.encoder = Encoder(num_layers, em_dim, num_heads, feed_dim,
                               input_vocab_size, pe_input, rate)

        # --- 2. 작가팀(Decoder) 고용 ---
        # 인코더의 분석본을 참고하여 번역문(타겟)을 생성하는 역할
        self.decoder = Decoder(num_layers, em_dim, num_heads, feed_dim,
                               target_vocab_size, pe_target, rate)

        # --- 3. 최종 출력층 (Linear) ---
        # 디코더가 내뱉은 '생각 벡터(em_dim)'를 '단어 점수표(vocab_size)'로 변환합니다.
        # 예: 512차원 벡터 -> 8000개 단어 중 누가 정답일 확률이 높은지 점수 매기기
        self.final_layer = nn.Linear(em_dim, target_vocab_size)

    # [핵심 기능: 마스크 생성 공장]
    # 훈련이나 생성 시 필요한 3가지 종류의 '가리개(Mask)'를 한 번에 만듭니다.
    def create_masks(self, inp, tar, pad_token=0):
        # 1. 인코더 패딩 마스크 (Encoder Padding Mask)
        # 인코더가 입력 문장(inp)을 분석할 때, [PAD](0) 부분은 쳐다보지 말라고 가립니다.
        enc_padding_mask = create_padding_mask(inp, pad_token)

        # 2. 디코더 패딩 마스크 (Decoder Padding Mask -> for Cross Attention)
        # ⭐️ 중요: 여기서 입력은 'inp'(원본 문장)입니다.
        # 디코더의 2차 회의(Cross-Attention)에서, 인코더의 결과물(K, V)을 참조할 때
        # 원본 문장의 [PAD] 부분은 무시하라고 알려주는 마스크입니다.
        dec_padding_mask = create_padding_mask(inp, pad_token)

        # 3. 룩어헤드 마스크 (Look-Ahead Mask -> for Masked Self Attention)
        # 디코더의 1차 회의(Self-Attention)에서 사용됩니다.
        seq_len = tar.size(1)
        # (1) 미래 가리기: 자기보다 뒤에 있는 단어는 못 보게 삼각형 마스크 생성
        look_ahead = create_look_ahead_mask(seq_len).to(tar.device)  # (seq_len, seq_len)
        look_ahead = look_ahead.unsqueeze(0).unsqueeze(0)  # 차원 맞추기 (1,1,seq_len,seq_len)

        # (2) 타겟 패딩 가리기: 정답지(tar) 자체에 있는 [PAD]도 못 보게 해야 함
        dec_target_padding_mask = create_padding_mask(tar, pad_token)  # (batch_size, 1, 1, seq_len)

        # (3) 결합: "미래도 가리고" AND "패딩도 가려야" 합니다.
        # combined_mask는 디코더가 '자기 자신'을 볼 때 사용됩니다.
        combined_mask = look_ahead & dec_target_padding_mask

        return enc_padding_mask, combined_mask, dec_padding_mask

    # [훈련(Training) 모드 동작]
    # inp: 원본 문장 ("I am a student")
    # tar: 정답 문장 ("나는 학생 입니다") -> 사실은 Shifted Right 된 입력 ([BOS], 나는, 학생)
    def forward(self, inp, tar, pad_token=0):
        """
        inp, tar: (batch_size, seq_len)
        """
        # 1. 필요한 3가지 마스크를 모두 생성합니다.
        enc_padding_mask, combined_mask, dec_padding_mask = self.create_masks(inp, tar, pad_token)

        # 2. 인코더 실행 (분석)
        # 원본 문장(inp)과 패딩 마스크를 주면, '최종 분석본(enc_output)'이 나옵니다.
        # enc_output shape: (batch_size, inp_seq_len, em_dim)
        enc_output = self.encoder(inp, enc_padding_mask)

        # 3. 디코더 실행 (생성 연습)
        # 정답지(tar), 인코더 분석본(enc_output), 그리고 두 종류의 마스크를 줍니다.
        # dec_output shape: (batch_size, tar_seq_len, em_dim)
        dec_output, attention_weights = self.decoder(
            tar, enc_output, look_ahead_mask=combined_mask, padding_mask=dec_padding_mask
        )

        # 4. 최종 단어 예측
        # 디코더의 출력을 단어 사전 크기로 변환하여 각 단어의 확률 점수(Logits)를 계산합니다.
        final_output = self.final_layer(dec_output)  # (batch_size, tar_seq_len, target_vocab_size)

        return final_output, attention_weights

    # [추론(Inference) 모드 동작: 텍스트 생성]
    # 정답지(tar) 없이, 모델이 스스로 문장을 만들어내는 과정입니다. (Autoregressive)
    # (셰프가 주문(enc_input)을 받고 요리를 완성해가는 과정과 같습니다.)
    @torch.no_grad() # 추론 시에는 학습(기울기 계산)을 안 하므로 메모리를 아낍니다.
    def generate(self, enc_input, start_tokens, end_token=None, pad_token=0, max_new_tokens=50, device='cpu'):
        '''
        enc_input : 번역할 원본 문장 (예: [1, 2, 3, 4] -> "I love you")
        start_tokens : 시작 신호 ID (예: [2] -> [BOS])
        '''

        # --- 1. 재료 준비 및 육수 끓이기 (인코더 실행) ---
        # 원본 문장(enc_input)을 GPU로 옮깁니다.
        enc_input = enc_input.to(device)

        # [중요] 인코더는 '딱 한 번'만 실행합니다!
        # 육수를 한 번 진하게 끓여두면 요리 끝날 때까지 계속 쓰는 것과 같습니다.
        # 원본 문장은 변하지 않으므로, 미리 분석해서 'enc_output(도서관/육수)'을 만들어 둡니다.
        enc_padding_mask = create_padding_mask(enc_input, pad_token)
        enc_output = self.encoder(enc_input, enc_padding_mask.to(device))

        # --- 2. 첫 숟가락 뜨기 (생성 시작) ---
        # 빈 그릇에 [BOS](시작 토큰) 하나를 딱 담아둡니다.
        # generated_tokens = [[BOS]]
        generated_tokens = [start_tokens]

        # --- 3. 맛보며 재료 추가하기 (디코더 반복 실행) ---
        # 단어를 하나씩 생성하는 루프입니다. (최대 50단어까지)
        for _ in range(max_new_tokens):

            # 현재까지 만든 문장(generated_tokens)을 모델에 넣기 위해 텐서로 변환합니다.
            # 예: [[BOS], "나는", "학교에"] -> shape: (1, 3)
            cur_tokens = torch.tensor(generated_tokens, dtype=torch.long, device=device)
            cur_tokens = cur_tokens.unsqueeze(0)  # 배치 차원 추가 (1, current_length)

            # 마스크 생성 (Combined Mask)
            # 디코더가 "지금까지 쓴 글"을 볼 때, 미래를 보지 못하게 가립니다.
            # (인코더 마스크는 위에서 만든거 재탕, 디코더 패딩 마스크는 현재 길이에 맞게 생성)
            enc_padding_mask, combined_mask, dec_padding_mask = self.create_masks(enc_input, cur_tokens, pad_token)

            # [디코더의 고민 시간]
            # 입력 1: cur_tokens ("지금까지 쓴 글")
            # 입력 2: enc_output ("아까 끓여둔 육수" / 원본 문장 정보)
            # 동작: "지금까지 '나는 학교에'라고 썼고, 원본 내용은 이거니까... 다음엔 무슨 말이 와야 하지?"
            with torch.no_grad():
                dec_output, _ = self.decoder(
                    cur_tokens, enc_output,
                    look_ahead_mask=combined_mask,
                    padding_mask=dec_padding_mask
                )

                # --- 4. 다음 재료 후보 점수 매기기 (단어 예측) ---
                # dec_output(생각 벡터)을 단어장 크기(8000개)의 점수표로 변환합니다.
                # logits shape: (1, 문장길이, 8000)
                logits = self.final_layer(dec_output)

                # [핵심] 우리는 문장의 '맨 마지막 단어'의 '다음 단어'만 궁금합니다.
                # [:, -1, :] -> 마지막 시점(time step)의 예측 점수만 쏙 빼옵니다.
                # 결과: "간다"(90점), "먹는다"(5점), "잔다"(1점)...
                next_token_logits = logits[:, -1, :]

                # --- 5. 최고의 재료 선택 (Greedy Search) ---
                # 점수가 가장 높은(argmax) 단어 번호를 하나 고릅니다.
                # "그래, '간다'가 점수가 제일 높네. 너로 정했다!"
                next_token = next_token_logits.argmax(dim=-1).item()

            # --- 6. 냄비에 추가 (결과 기록) ---
            # 선택된 단어('간다')를 문장 리스트에 이어 붙입니다.
            # generated_tokens: [[BOS], "나는", "학교에"] -> [[BOS], "나는", "학교에", "간다"]
            # (이 리스트는 다음 루프에서 다시 'cur_tokens'가 되어 디코더에 들어갑니다.)
            generated_tokens.append(next_token)

            # --- 7. 요리 완성 확인 (종료 조건) ---
            # 만약 모델이 [EOS](끝) 토큰을 꺼냈다면?
            # "문장이 다 완성되었습니다." 하고 루프를 멈춥니다.
            if end_token is not None and next_token == end_token:
                break

        # 완성된 전체 문장(토큰 리스트)을 반환합니다.
        return generated_tokens

In [ ]:
# --- 트랜스포머 모델 하이퍼파라미터 설정 ---
# (이 코드를 실행하려면 'torch'와 'Transformer' 클래스 및
# 'positional_encoding', 'Encoder', 'Decoder' 등 모든 하위 모듈이
# 이전에 정의되어 있어야 합니다.)
import torch
import torch.nn as nn
import math

# (이전 정의들이 여기에 있다고 가정...)
# def positional_encoding(...): ...
# class MultiHeadAttention(nn.Module): ...
# def point_wise_feed_forward_network(...): ...
# class EncoderLayer(nn.Module): ...
# class DecoderLayer(nn.Module): ...
# class Encoder(nn.Module): ...
# class Decoder(nn.Module): ...
# class Transformer(nn.Module): ...
# def create_padding_mask(...): ...
# def create_look_ahead_mask(...): ...


# 인코더 입력 예시 (배치 1, 시퀀스 10) / <pad> 토큰 0 가정
# 'eco_inp': (Encoder Input) "I am a student" (원본 문장)
# (1~6은 단어 ID, 0은 [PAD] 토큰을 의미합니다.)
eco_inp = torch.Tensor([[1, 2, 3, 4, 5, 6, 0, 0, 0, 0]]).long()

# 디코더 입력 예시 (배치 1, 시퀀스 10) / <pad> 토큰 0 가정
# 'dco_inp': (Decoder Input) " [BOS] 나는 학생 입니다 [EOS]" (정답 문장, Shifted Right)
# (이것은 '훈련(forward)' 시에 '교사 강요'용 힌트로 사용될 '정답지'입니다.)
dco_inp = torch.Tensor([[1, 2, 3, 4, 5, 6, 7, 8, 0, 0]]).long()


# 🧪 트랜스포머 모델 실행 및 결과 분석
# 이 코드는 우리가 지금까지 단계별로 조립해 온 '트랜스포머(Transformer)' 모델을 실제로 작동시켜 보는 최종 테스트 단계입니다.
# 마치 자동차 공장에서 엔진, 바퀴, 차체를 다 조립한 뒤, 시동을 걸어보고(forward), 자율주행 모드도 켜보는(generate) 것과 같습니다.

# --- 1. 하이퍼파라미터 설정 (자동차 스펙 정하기) ---
# 모델의 성능과 크기를 결정하는 설정값들입니다. 테스트용이라 작게 설정되어 있습니다.

num_layers = 2        # 인코더와 디코더를 각각 2층씩 쌓음 (2학년 작가팀)
em_dim = 32           # 단어 하나를 32개의 숫자로 표현 (생각의 깊이)
num_heads = 4         # 4명의 전문가가 병렬로 분석 (멀티 헤드)
feed_dim = 64         # 생각 정리(FFN) 시 64차원까지 확장해서 고민함
input_vocab_size = 16   # 입력 언어 단어장 크기 (단어 16개만 아는 모델)
target_vocab_size = 16  # 출력 언어 단어장 크기
pe_input = 100        # 입력 문장은 최대 100단어까지 가능
pe_target = 100       # 출력 문장도 최대 100단어까지 가능

# --- 2. 모델 생성 (공장 가동) ---
# 설정된 스펙대로 트랜스포머 객체(transformer)를 생성합니다.
# 이제 인코더와 디코더가 모두 준비되었습니다.
transformer = Transformer(num_layers, em_dim, num_heads, feed_dim,
                      input_vocab_size, target_vocab_size,
                      pe_input, pe_target)

# --- 3. 훈련 모드 실행 (forward) - "정답지 보고 연습하기" ---
# 'transformer' 객체의 'forward' 함수를 호출합니다.
# eco_inp: 인코더 입력 (원본 문장)
# dco_inp: 디코더 입력 (정답 문장, Shifted Right)
# 동작: 인코더가 eco_inp를 분석하고, 디코더가 dco_inp를 힌트 삼아 다음 단어를 예측합니다.
out, atw = transformer(eco_inp, dco_inp)

# --- 4. 생성 모드 실행 (generate) - "실전 번역하기" ---
# 'transformer' 객체의 'generate' 함수를 호출합니다.
# eco_inp: 원본 문장
# 2: 시작 토큰 [BOS] (여기서부터 시작해라)
# 3: 종료 토큰 [EOS] (이게 나오면 멈춰라)
# 0: 패딩 토큰 (마스크 생성 시 무시할 ID)
generated_token = transformer.generate(eco_inp, 2, 3, 0)

# --- 결과 출력 ---

# 'out' 텐서의 모양(shape)을 출력합니다.
# 결과 (out): torch.Size([1, 10, 16])
# 1: 배치 크기 (문장 1개)
# 10: 시퀀스 길이 (디코더 입력(dco_inp)이 10개였으므로, 10개의 각 위치에서 다음 단어를 예측)
# 16: 단어장 크기 (각 10개 위치마다 16개 단어 중 무엇일 확률이 높은지 '점수표'를 출력)
print(f'최종 출력 형상 {out.shape}')

# 'generate' 함수가 반환한 '토큰 ID 리스트'를 출력합니다.
# 결과 (generated_token): [2, 0, 8, 0, 5, ..., 15, 3] (예시)
# [2]: 시작 토큰(start_token=2)으로 시작했습니다.
# [..., 3]: 마지막에 종료 토큰(end_token=3)이 나와서 생성이 끝났습니다.
# (참고): 현재 모델은 학습(훈련)이 전혀 안 된 상태(무작위 초기화 상태)라서,
# 생성된 단어 배열(0, 8, 0, 5...)은 아무 의미 없는 '무작위 값'입니다.
# (실제 훈련을 시키면 의미 있는 문장이 나옵니다.)
print(f'토큰 생성 {generated_token}')

# 🏁 최종 결론
# 이 코드가 에러 없이 실행되었다는 것은, 인코더, 디코더, 어텐션, 마스크 등
# 모든 복잡한 부품들이 완벽하게 맞물려 돌아가고 있음을 의미합니다.
# 이제 데이터를 넣어 학습(optimizer.step())만 시키면
# 훌륭한 번역기가 될 준비를 마친 것입니다.

최종 출력 형상 torch.Size([1, 10, 16])
토큰 생성 [2, 0, 8, 0, 5, 5, 5, 5, 5, 5, 5, 15, 5, 5, 5, 5, 5, 5, 15, 5, 5, 5, 5, 15, 3]


## Transformer 학습 하기


### 데이터세트 준비

이전 Seq2seq 모델 학습과 동일하게 준비한 Transformer 모델에 [AIHUB](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=86) 한국어 감성 대화 말뭉치를 부분적으로 활용하여 질의(입력) 응답(타겟) 데이터세트를 학습 합니다.

In [ ]:
# [전체 흐름]
# 이 코드는 이전의 'Seq2seq + Attention (LSTM)'과는 '다른 방식'으로
# '토크나이저'와 '데이터셋'을 준비하는 과정입니다.
#
# 1. (파일 업로드): 'train.csv' 파일을 Colab 환경으로 업로드합니다.
# 2. (CSV 읽기): 'pandas'로 'train.csv'를 읽어 'df' 변수에 저장하고 내용을 확인합니다.
# 3. (코퍼스 생성): 'train.csv'의 'HS01'(질문) 열 '만'을 'train.txt' 파일로 저장합니다.
#    (주의! 이전 코드와 달리 'SS01'(답변)은 코퍼스에 '포함하지 않습니다'.
#     이는 토크나이저가 '답변'에만 쓰이는 단어를 '학습하지 못할' 수 있음을 의미합니다.)
# 4. (SPM 훈련): 'train.txt'(질문만 있음)를 바탕으로 'vocab_size=2000'짜리
#    'spm_krsent.model' 토크나이저를 'model' 폴더 안에 생성합니다.
#    (주의! 이전과 달리 `pad_id`, `bos_id`, `eos_id`를 '지정하지 않았습니다'.
#     -> SentencePiece 기본값: UNK=0, BOS=1, EOS=2, PAD=미지정(-1))
# 5. (SPDataSet 정의): Seq2seq 모델에 맞는 'Dataset' 클래스를 정의합니다.
#    (주의! `zero_pad` 함수는 `np.zeros`를 사용하므로 'PAD=0'이라고 '가정'합니다.
#     이는 SPM 훈련(PAD=-1)과 '불일치'하며, 'UNK=0'과도 '충돌'합니다. -> 코드 설계 오류 가능성)
# 6. (데이터로더 생성): 훈련된 'sp'와 'SPDataSet'으로 'dataloader'를 만듭니다.
# 7. (데이터 확인): 'dataloader'에서 '1개 배치'를 꺼내어
#    'inp' (인코더 입력), 'tar[:,:-1]' (디코더 입력), 'tar[:,1:]' (디코더 정답)
#    3가지 형태의 텐서를 출력해봅니다.

# google.colab 환경에서 파일을 업로드할 수 있는 'files' 모듈을 가져옵니다.
from google.colab import files
# 'files.upload()'가 실행되면, 사용자의 컴퓨터에서 파일을 선택할 수 있는 업로드 창이 뜹니다.
# (이 코드를 실행하려면 'train.csv' 파일을 업로드해야 합니다.)
uploaded = files.upload()

Saving train.csv to train (1).csv


In [ ]:
# 파이토치 'Dataset', 'DataLoader', 'random_split'을 가져옵니다.
from torch.utils.data import Dataset, DataLoader, random_split
# 'sentencepiece' 라이브러리를 'spm'이라는 별칭으로 가져옵니다.
import sentencepiece as spm
# 'pandas' 라이브러리를 'pd'라는 별칭으로 가져옵니다.
import pandas as pd
# (SPDataSet에서 'np'를 사용하므로, 'import numpy as np'가 사실 필요합니다.)
import numpy as np
# (SPDataSet에서 'torch.Tensor'를 반환하므로, 'import torch'가 사실 필요합니다.)
import torch

# 'train.csv' 파일을 'pd.read_csv'로 읽어 'df' 변수(DataFrame)에 저장합니다.
df = pd.read_csv(f'./train.csv')
# 'df'의 내용을 (Colab 환경에서 표로) 출력합니다.
df

,label,HS01,SS01,HS02,SS02,HS03,SS03
0,E1,일은 왜 해도 해도 끝이 없을까? 화가 난다.,많이 힘드시겠어요. 주위에 의논할 상대가 있나요?,그냥 내가 해결하는 게 나아. 남들한테 부담 주고 싶지도 않고.,혼자 해결하기로 했군요. 혼자서 해결하기 힘들면 주위에 의논할 사람을 찾아보세요.,NaN,NaN
1,E1,이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.,급여가 줄어 속상하시겠어요. 월급이 줄어든 것을 어떻게 보완하실 건가요?,최대한 지출을 억제해야겠어. 월급이 줄어들었으니 고정지출을 줄일 수밖에 없을 것 같아.,월급이 줄어든 만큼 소비를 줄일 계획이군요.,NaN,NaN
2,E1,회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...,회사 동료 때문에 스트레스를 많이 받는 것 같아요. 문제 해결을 위해 어떤 노력을 ...,잘 안 맞는 사람이랑 억지로 잘 지내는 것보단 조금은 거리를 두고 예의를 갖춰서 대...,스트레스받지 않기 위해선 인간관계에 있어 약간의 거리를 두는 게 좋겠군요.,NaN,NaN
3,E1,직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...,관련 없는 심부름을 모두 하게 되어서 노여우시군요. 어떤 것이 상황을 나아질 수 있...,직장 사람들과 솔직하게 이야기해보고 싶어. 일하는 데에 방해된다고.,직장 사람들과 이야기를 해 보겠다고 결심하셨군요.,NaN,NaN
4,E1,얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.,무시하는 것 같은 태도에 화가 나셨군요. 상대방의 어떤 행동이 그런 감정을 유발하는...,상사인 나에게 먼저 인사하지 않아서 매일 내가 먼저 인사한다고!,항상 먼저 인사하게 되어 화가 나셨군요. 어떻게 하면 신입사원에게 화났음을 표현할 ...,NaN,NaN
...,...,...,...,...,...,...,...
51623,E1,나이가 먹고 이제 돈도 못 벌어 오니까 어떻게 살아가야 할지 막막해. 능력도 없고.,경제적인 문제 때문에 막막하시군요. 마음이 편치 않으시겠어요.,아무것도 할 수 없는 내가 무가치하게 느껴지고 실망스러워.,지금 할 수 있는 가장 합리적인 행동은 무엇인가요?,노년층을 위한 경제적 지원이나 부업 같은 것도 알아보아야겠어.,좋은 결과 얻으시길 바랄게요.
51624,E3,몸이 많이 약해졌나 봐. 이제 전과 같이 일하지 못할 것 같아 너무 짜증 나.,건강에 대한 어려움 때문에 기분이 좋지 않으시군요. 속상하시겠어요.,마음 같아서는 다 할 수 있는 일인데 이젠 몸이 안 따라와 주니 화만 나.,어떻게 하면 지금의 기분을 나아지게 할 수 있을까요?,남편과 함께 게이트볼이나 치러 가야겠어. 그럼 기분이 나아질 것 같아.,남편과 함께하는 좋은 외출 시간 되시길 바랄게요.
51625,E4,이제 어떻게 해야 할지 모르겠어. 남편도 그렇고 노후 준비도 안 되어서 미래가 걱정돼.,노후 준비에 대한 어려움 때문에 걱정이 많으시겠어요.,주변 사람들은 다 노후 준비도 잘해두었던데 난 어떻게 해야 할지 모르겠어. 막막하기...,지금의 상황에서 할 수 있는 가장 좋은 행동이 무엇일까요?,남편과 함께 실버 일자리나 노년층을 위한 국가 지원에 대해 자세히 알아보아야겠어.,좋은 정보 많이 얻으셔서 걱정을 좀 덜으셨으면 좋겠어요.
51626,E3,몇십 년을 함께 살았던 남편과 이혼했어. 그동안의 세월에 배신감을 느끼고 너무 화가 나.,가족과의 문제 때문에 속상하시겠어요.,이제 할 수 있는 일도 없고 이렇게 힘들게 사는 게 불만스럽기만 해.,지금의 감정을 나아지게 할 수 있는 어떤 방법이 있을까요?,함께 친하게 지내던 동네 언니 동생들과 빈자리를 조금이나마 채울까 해.,지인분들과 좋은 시간 보내셨으면 좋겠어요.


In [ ]:
# 'sentencepiece', 're' (정규표현식, 여기선 사용 안 됨), 'os' (파일/디렉터리) 라이브러리를 가져옵니다.
import sentencepiece as spm
import re
import os

# [지금 하는 일: 훈련용 코퍼스 파일 생성]
# 'train.txt' 파일을 'a' (append, '추가 이어쓰기') 모드로 엽니다.
# (만약 'train.txt'가 이미 존재하고 내용이 있었다면, 그 '뒤에' 텍스트가 추가됩니다.)
# (보통은 'w' (write, '덮어쓰기') 모드를 쓰는 것이 더 안전합니다.)
with open('train.txt', 'a', encoding='utf-8') as f:
    # 'df' (DataFrame)에서 'HS01'(질문) 열의 '모든' 텍스트를 하나씩 반복합니다.
    # (주의! 'SS01'(답변) 열은 이 루프에서 사용(학습)되지 않습니다.)
    for text in df['HS01'] :
        # 'text'가 'nan' (Not a Number, 빈 값) 등 문자열이 아닐 경우를 대비해 'str()'로 강제 변환합니다.
        text = str(text)
        # try...except: 'f.write' 과정에서 (드물지만) 인코딩 오류 등이 발생할 경우,
        try:
            # 'text' (질문 1개)를 파일(f)에 쓰고, '줄바꿈(\n)'을 추가합니다.
            f.write(text+'\n')
        except:
            # 오류가 발생하면, 'pass' (그냥 아무것도 안 하고) 넘어갑니다.
            pass


# 'os.makedirs'를 사용해 './model'이라는 이름의 디렉터리를 생성합니다.
# exist_ok=True: 'model' 디렉터리가 이미 존재하더라도 오류를 발생시키지 않고 넘어갑니다.
os.makedirs('./model', exist_ok=True)

# [지금 하는 일: SentencePiece 토크나이저 훈련]
# 'spm.SentencePieceTrainer.train' 함수를 호출하여 '토크나이저 모델'을 훈련합니다.
spm.SentencePieceTrainer.train(
    # input: 훈련시킬 '코퍼스' 파일. (방금 만든 'train.txt')
    input='train.txt',
    # model_prefix: 훈련된 모델을 저장할 '경로'와 '파일 접두사'.
    # (-> './model/spm_krsent.model'과 './model/spm_krsent.vocab' 파일이 생성됨)
    model_prefix='./model/spm_krsent',
    # vocab_size: 토크나이저가 만들 '단어 집합(어휘)'의 최대 크기를 '2000'개로 설정합니다.
    vocab_size=2000
    # (주의! pad_id=0, bos_id=1, eos_id=2 등을 '지정하지 않았습니다'.)
    # (-> SPM 기본값: UNK=0, BOS=1, EOS=2)
)

In [ ]:
# 'SPDataSet'이라는 이름의 새 클래스를 정의합니다. PyTorch의 'Dataset'을 상속받습니다.
class SPDataSet(Dataset):
    # 초기화(생성자) 함수. (sp: 토크나이저 객체, max_len: 최대 길이)
    def __init__(self, sp, max_len):
        # 'max_len' (예: 60)을 'self.max_len' 변수에 저장합니다.
        self.max_len = max_len
        # 'train.csv' 파일을 (다시) 읽어와서 'HS01'(질문), 'SS01'(답변) 열만 'self.df'에 저장합니다.
        self.df = pd.read_csv(f'./train.csv')[['HS01','SS01']]
        # 'sp' (토크나이저 객체)를 'self.sp' 변수에 저장합니다.
        self.sp = sp

    # 'zero_pad' (패딩/절단) 함수를 정의합니다.
    def zero_pad(self, tok):
        # 만약 토큰 리스트(tok)의 길이가 'max_len'보다 길거나 같다면,
        if len(tok) >= self.max_len:
            # 'max_len'까지만 잘라내어 반환합니다.
            return tok[:self.max_len]
        # 만약 토큰 리스트의 길이가 'max_len'보다 짧다면,
        else:
            # 'np.zeros'를 사용해 'max_len' 길이만큼 '0'으로 채워진 numpy 배열을 만듭니다.
            # (주의! 이것은 'PAD' 토큰의 ID가 '0'이라고 '가정'하는 것입니다.)
            # (SPM 훈련 시 PAD를 지정 안 했으므로, 'UNK=0'과 'PAD=0'이 '충돌'합니다.)
            padding = np.zeros(self.max_len)
            # 0 배열의 '앞부분'을 실제 토큰 리스트(tok) 내용으로 덮어씁니다.
            padding[:len(tok)] = tok
            # 패딩된 배열을 반환합니다.
            return padding

    # 데이터셋의 '총 개수'를 반환하는 함수입니다.
    def __len__(self):
        # 'self.df' (DataFrame)의 전체 행(row)의 개수(len)를 반환합니다.
        return (len(self.df))

    # 'i'번째 데이터를 요청받았을 때, '가공된 텐서'를 반환하는 함수입니다.
    def __getitem__(self, i):
        # 'self.df' (DataFrame)에서 'i'번째 행('iloc[i]')을 가져옵니다.
        sent = self.df.iloc[i]
        # 'i'번째 행의 'HS01'(질문) 텍스트를 'sp'로 '토큰 ID 리스트'로 변환합니다.
        sent1 = self.sp.encode_as_ids(sent['HS01'])
        # 'i'번째 행의 'SS01'(답변) 텍스트를 'sp'로 '토큰 ID 리스트'로 변환합니다.
        sent2 = self.sp.encode_as_ids(sent['SS01'])

        # [인코더 입력 (inp) 생성]
        # 'sent1'(질문)에 'self.sp.eos_id()'(문장 끝, ID 2)를 '추가'합니다.
        # (이유? 일부 트랜스포머 구현은 인코더 입력에도 [EOS]를 붙여 "여기서 질문 끝"을 명시합니다.)
        # 'zero_pad'로 패딩/절단합니다.
        inp = self.zero_pad(sent1 + [self.sp.eos_id()])

        # [디코더 타겟 (tar) 생성]
        # 'sent2'(답변)의 '앞'에 'self.sp.bos_id()'(문장 시작, ID 1)를,
        # '뒤'에 'self.sp.eos_id()'(문장 끝, ID 2)를 '추가'합니다.
        # (이유? [BOS]는 "이제 생성 시작해"라는 '첫 힌트'로, [EOS]는 "여기서 멈춰"라는 '정답'으로 필요합니다.)
        # 'zero_pad'로 패딩/절단합니다.
        tar = self.zero_pad([self.sp.bos_id()] + sent2 + [self.sp.eos_id()])

        # 'inp'(질문 텐서)와 'tar'(답변 텐서)를 반환합니다.
        return torch.Tensor(inp), torch.Tensor(tar)


# [지금 하는 일: 데이터로더 생성 및 확인]
# 'spm.SentencePieceProcessor' 객체를 생성하고, 훈련된 './model/spm_krsent.model' 파일을 '불러옵니다'.
sp = spm.SentencePieceProcessor(model_file=f'./model/spm_krsent.model')
# 'SPDataSet' 객체를 생성합니다. (토크나이저 'sp', 최대 길이 '60' 전달)
dataset = SPDataSet(sp, 60)
# 'DataLoader' (데이터 공급기)를 생성합니다. (배치 크기 1, 매번 섞음)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# 'dataloader'에서 '1개 배치'(inp, tar)를 꺼냅니다.
for inp, tar in dataloader:
    # --- 1. 'inp' (인코더 입력) ---
    # (모양: [1, 60])
    # (내용: [질문단어1, ..., 질문단어N, EOS, PAD, ...])
    print(inp.long())

    # [지금 하는 일: (2) '디코더 입력' (힌트) 생성 및 출력]
    # 'tar' (원본 답변, [BOS], w1, w2, ..., EOS, PAD) 텐서에서
    # '[:, :-1]' (모든 배치(':')에 대해, '마지막 1개([-1])'를 '제외'하고(:-1) 모두 가져옴)
    #
    # (원본 tar): [BOS, w1, w2, ..., EOS, PAD]
    # (결과):     [BOS, w1, w2, ..., EOS]
    #
    # (이것이 'Outputs (shifted right)'이며, 디코더의 '입력(힌트)'으로 사용됩니다.)
    # (모양: [1, 59])
    print(tar.long()[:,:-1])

    # [지금 하는 일: (3) '디코더 정답' (타겟) 생성 및 출력]
    # 'tar' (원본 답변, [BOS], w1, w2, ..., EOS, PAD) 텐서에서
    # '[:, 1:]' (모든 배치(':')에 대해, '첫 번째 1개([BOS])'를 '제외'하고(1:) 모두 가져옴)
    #
    # (원본 tar): [BOS, w1, w2, ..., EOS, PAD]
    # (결과):     [w1, w2, ..., EOS, PAD]
    #
    # (이것이 디코더가 '매 스텝 예측해야 할 실제 정답(Label)'입니다.)
    # (모양: [1, 59])
    print(tar.long()[:,1:])

    # [왜 이렇게 짝을 맞추나? (교사 강요)]
    # (t=1) 입력 `[BOS]` (2번 결과) -> 정답 `[w1]` (3번 결과)
    # (t=2) 입력 `[w1]` (2번 결과) -> 정답 `[w2]` (3번 결과)
    # ...
    # (t=N) 입력 `[EOS]` (2번 결과) -> 정답 `[PAD]` (3번 결과)
    # (이렇게 '입력'과 '정답'이 '한 칸씩 밀린(shifted)' 짝이 완성되어 훈련이 가능해집니다.)

    # 테스트를 위해 '1개 배치'만 확인하고 '즉시 중단'합니다.
    break

# --- 출력 결과 분석 ---
# (첫 번째 텐서: 'inp')
# tensor([[  18,   66,    5, ...,    4,    2,    0, ...]])
# (해석: [단어들...]이 나오고, [EOS](ID 2)로 끝난 뒤, [PAD](ID 0)가 채워져 있습니다.)

# (두 번째 텐서: 'tar[:, :-1]' (디코더 입력 힌트))
# tensor([[   1,   66,   29, ...,    4,    2]])
# (해석: [BOS](ID 1)로 '시작'하고, [EOS](ID 2)로 '끝납니다'. (마지막 PAD가 잘려나감))

# (세 번째 텐서: 'tar[:, 1:]' (디코더 예측 정답))
# tensor([[  66,   29,  280, ...,    2,    0, ...]])
# (해석: [BOS] 없이 '첫 단어(66)'로 '시작'하고, [EOS](ID 2)를 거쳐 [PAD](ID 0)로 '끝납니다'.)

tensor([[  18,   66,    5,   54,  999,   29,  280,  825,    7,  923,  252,   46,
            3, 1971,  789,  338,  340,    4,    2,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0]])
tensor([[   1,   66,   29,  280,  825,  923, 1319,   12,  429,  270, 1888,  826,
           14,  520,  433,    4,    2,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0]])
tensor([[  66,   29,  280,  825,  923, 1319,   12,  429,  270, 1888,  826,   14,
          520,  433,    4,    2,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    

### 모델 학습

Transformer 구조는 Seq2seq와 다르게 디코더가 타겟 토큰을 한번에 학습하고 예측된 토큰을 시퀀스 길이만큼 반환합니다.  
반환된 시퀀스 길이의 예측 토큰들을 한번에 손실 연산 하기 위해 배치크기와 시퀀스 길이를 곱하여 형상을 변환합나다.



In [ ]:
# (이 코드를 실행하려면 'torch', 'nn', 'Transformer', 'SPDataSet' 등이 이전 코드에서 정의되어 있어야 합니다.)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
# (sentencepiece 라이브러리 필요)
import sentencepiece as spm

# [전체 흐름: 트랜스포머 모델 훈련 실행]
# 이 코드는 정의된 'Transformer' 모델을 실제로 '훈련(Training)'시키는 메인 스크립트입니다.
# "받아쓰기 시험" 비유를 통해 설명하겠습니다.
#
# 1. (준비물): 채점 기준표(손실 함수), 하이퍼파라미터(설정값), 학생(모델), 선생님(옵티마이저), 문제집(데이터셋)을 준비합니다.
# 2. (반복 학습): 총 50번(epochs) 문제집을 처음부터 끝까지 풉니다.
# 3. (실전 풀이):
#    - 64문제씩(batch) 풉니다.
#    - 학생이 답안(pred)을 작성합니다.
#    - 선생님이 정답(tar_out)과 비교해 점수(loss)를 매깁니다.
#    - 틀린 만큼 학생의 뇌(가중치)를 수정(backprop)합니다.
# 4. (경과 보고): 200페이지마다, 그리고 한 권 다 풀 때마다 성적(Loss)을 알려줍니다.

# --- 1. 손실 함수 정의 (채점 기준표) ---
# batch_size, seq_len => batch_size * seq_len 하여 손실 연산
def loss_function(real, pred, pad_token=0):
    # (주석) pred: 학생이 써낸 답안지. (확률 점수표) [B, S, Vocab]
    # (주석) real: 실제 정답지. (단어 ID) [B, S]

    # 'nn.CrossEntropyLoss': "정답(ID)과 예측(확률)이 얼마나 다른지" 계산하는 채점기입니다.
    # (ignore_index=pad_token): "[PAD](0)는 채점하지 마세요. (빈칸이니까 틀려도 됨)"
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token)

    # (모양 변경) 3D 텐서를 2D로 '펼칩니다'.
    # pred: [B, S, V] -> [B*S, V] (예: 64문장*60단어 = 3840개의 단어 예측 문제로 취급)
    pred_reshaped = pred.view(-1, pred.size(-1))
    # real: [B, S] -> [B*S] (3840개의 정답)
    real_reshaped = real.reshape(-1)

    # 펼쳐진 문제들에 대해 손실을 계산하고 반환합니다.
    return loss_fn(pred_reshaped, real_reshaped)

# --- 2. 하이퍼파라미터 설정 (훈련 환경 설정) ---
# 토크나이저 로딩
sp = spm.SentencePieceProcessor(model_file=f'./model/spm_krsent.model')

# 모델 크기 설정
num_layers = 4      # 4층짜리 모델 (4학년)
d_model = 128       # 생각의 깊이 (128차원)
dff = 256           # 심화 학습 확장 크기 (256차원)
num_heads = 4       # 전문가 수 (4명)
vocab_size = sp.get_piece_size() # 단어장 크기 (2000개)

# 위치 인코딩 최대 길이
pe_input = 10000
pe_target = 10000

# 학습 관련 설정
dropout_rate = 0.1  # 10% 드롭아웃
max_len = 60        # 문장 최대 길이 60
pad_token = 0       # 패딩 ID 0
lr = 2e-4           # 학습률 (공부 속도)
batch_size = 64     # 한 번에 64문제 풀기
epochs = 50         # 문제집 50번 반복 풀기

# GPU 설정 (GPU가 있으면 쓰고, 없으면 CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 3. 모델 및 최적화 도구 생성 ---
# 트랜스포머 모델(학생) 생성 및 GPU 탑승 (.to(device))
transformer = Transformer(num_layers, d_model, num_heads, dff,
                            vocab_size, vocab_size,
                            pe_input, pe_target, dropout_rate).to(device)

# 옵티마이저(선생님) 생성 (Adam)
# (transformer.parameters(): 학생의 모든 뇌세포(가중치)를 관리 대상으로 등록)
optimizer = torch.optim.Adam(transformer.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)

# --- 4. 데이터 준비 ---
# (주석) 간략한 실험을 위해 평가데이터 분리 생략
# SPDataSet (데이터셋) 생성
dataset = SPDataSet(sp, max_len)
# DataLoader (데이터 공급기) 생성 (64개씩 묶어서, 매번 섞어서 줌)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# --- 5. 훈련 루프 (공부 시작!) ---
# 50번 반복 (에포크)
for e in range(epochs):
    # 모델을 '훈련 모드'로 설정 (드롭아웃 켜기)
    transformer.train()
    # 이번 에포크의 '총 오답 점수' 초기화
    total_loss = 0.0

    # [이 부분만 집중 해부: 정답지 쪼개기 (Slicing)]
    #
    # 상황: "나는 학생입니다"라는 문장을 훈련시키려 합니다.
    # 토크나이저가 만든 '완벽한 정답지(tar)'는 숫자로 되어 있지만, 알기 쉽게 한글로 표현하면 이렇습니다.
    #
    # tar (원본) = [ [BOS], "나는", "학생", "입니다", [EOS] ]  (총 5개 토큰)
    #
    # 트랜스포머 디코더 훈련은 "앞 단어를 보고 뒷 단어 맞추기" 게임입니다.
    # 그래서 이 'tar' 하나를 '문제지(Input)'와 '정답지(Label)' 두 개로 쪼개야 합니다.

    # 1. 데이터 로더에서 꺼내기
    # inp: "I am a student" (영어 질문)
    # tar: [ [BOS], "나는", "학생", "입니다", [EOS] ] (완벽한 한글 답변)
    for i, (inp, tar) in enumerate(dataloader):

        # 데이터를 GPU로 옮깁니다. (계산 준비)
        inp = inp.long().to(device)
        tar = tar.long().to(device)

        # --- [핵심] 문제지 만들기 (tar_in) ---
        # 슬라이싱 [:, :-1]의 의미:
        #   - ':' (모든 배치 문장에 대해)
        #   - ':-1' (처음부터 ~ '맨 마지막 1개'를 빼고 다 가져와라)
        #
        # (원본 tar) : [ [BOS], "나는", "학생", "입니다", [EOS] ]
        # (tar_in)   : [ [BOS], "나는", "학생", "입니다" ]      <-- [EOS]가 잘림!
        #
        # (이유) 디코더에게 "자, 여기 힌트 줄게. 다음 단어 맞춰봐"라고 줄 '입력값'입니다.
        # 마지막 [EOS] 뒤에는 더 이상 맞출 단어가 없으니, 입력으로 줄 필요가 없습니다.
        tar_in = tar[:, :-1]

        # --- [핵심] 정답지 만들기 (tar_out) ---
        # 슬라이싱 [:, 1:]의 의미:
        #   - ':' (모든 배치 문장에 대해)
        #   - '1:' ('맨 앞의 1개'를 빼고 ~ 끝까지 다 가져와라)
        #
        # (원본 tar) : [ [BOS], "나는", "학생", "입니다", [EOS] ]
        # (tar_out)  : [        "나는", "학생", "입니다", [EOS] ]  <-- [BOS]가 잘림!
        #
        # (이유) 모델이 '실제로 맞춰야 할' 정답입니다.
        # 모델은 [BOS]를 보고 -> "나는"을 맞춰야 하고,
        #       "입니다"를 보고 -> [EOS]를 맞춰야 합니다.
        # [BOS]는 '주어지는 힌트'이지 '맞춰야 할 정답'이 아니므로 정답지에서 뺍니다.
        tar_out = tar[:, 1:]

        # [결과적으로 이렇게 짝이 지어집니다 (Teacher Forcing)]
        #
        # (모델에게 주는 힌트 tar_in)      (모델이 맞춰야 할 정답 tar_out)
        # -------------------------------------------------------
        #          [BOS]           --->        "나는"
        #         "나는"           --->        "학생"
        #         "학생"           --->        "입니다"
        #        "입니다"          --->        [EOS]
        #
        # 이렇게 '한 칸씩 밀려서(Shifted)' 짝꿍이 딱 맞게 됩니다.
        # 이래서 이름이 'Shifted Right'입니다.

        # 기울기 초기화 (오답노트 지우기)
        optimizer.zero_grad()

        # [모델 실행]
        # 학생(transformer)에게 '질문(inp)'과 '힌트(tar_in)'를 줍니다.
        # pred: 학생의 예측 답안
        # _: 어텐션 가중치는 훈련 중엔 필요 없어서 무시
        pred, _ = transformer(inp, tar_in, pad_token=pad_token)

        # [채점]
        # 학생 답안(pred)과 진짜 정답(tar_out)을 비교해 점수(loss)를 매깁니다.
        loss = loss_function(tar_out, pred, pad_token=pad_token)

        # 총점 누적 (기록용)
        total_loss += loss.item()

        # [복습/수정]
        # 역전파 (어디서 틀렸는지 찾기)
        loss.backward()
        # 가중치 업데이트 (뇌세포 수정)
        optimizer.step()

        # 200 배치마다 중간 성적 출력
        if (i+1) % 200 == 0:
            print(f"Epoch: {e}, Batch: {i+1}, Loss: {total_loss/(i+1)}")

    # 한 에포크(문제집 1권) 끝날 때마다 최종 성적 출력
    print(f"====>Epoch: {e}, Loss: {total_loss/(i+1)}")

# [출력 결과 분석]
# Epoch 0 -> Loss 4.08: 처음엔 4점대로 많이 틀립니다.
# Epoch 49 -> Loss 1.51: 50번 반복하니 1.5점대로 오답률이 확 줄었습니다. (학습 성공!)

Epoch: 0, Batch: 200, Loss: 5.200121765136719
Epoch: 0, Batch: 400, Loss: 4.6147064781188964
Epoch: 0, Batch: 600, Loss: 4.299499713182449
Epoch: 0, Batch: 800, Loss: 4.092008906304836
====>Epoch: 0, Loss: 4.085637493499888
Epoch: 1, Batch: 200, Loss: 3.2925089275836945
Epoch: 1, Batch: 400, Loss: 3.236822310090065
Epoch: 1, Batch: 600, Loss: 3.1873928618431093
Epoch: 1, Batch: 800, Loss: 3.146705023944378
====>Epoch: 1, Loss: 3.1454577339626155
Epoch: 2, Batch: 200, Loss: 2.940320063829422
Epoch: 2, Batch: 400, Loss: 2.911688976287842
Epoch: 2, Batch: 600, Loss: 2.8843137228488924
Epoch: 2, Batch: 800, Loss: 2.8610846373438834
====>Epoch: 2, Loss: 2.860541569108266
Epoch: 3, Batch: 200, Loss: 2.7388914477825166
Epoch: 3, Batch: 400, Loss: 2.7181621623039245
Epoch: 3, Batch: 600, Loss: 2.6974220196406047
Epoch: 3, Batch: 800, Loss: 2.6788124108314513
====>Epoch: 3, Loss: 2.6787941098360917
Epoch: 4, Batch: 200, Loss: 2.5630456447601317
Epoch: 4, Batch: 400, Loss: 2.5610054504871367
Epo

> Seq2seq 모델에 비해 더욱 빠른 속도로 손실이 줄어드는것을 확인할 수 있습니다. 충분한 학습이 이루어지면 적절한 텍스트를 생성할 수 있습니다.

### 생성 텍스트 평가하기

- BLEU(Bilingual Evaluation Understudy)
    * n-gram 기반으로 참조 문장과 생성 문장 간 **정밀도(Precision)**를 측정하며, 짧은 문장 페널티(Brevity Penalty) 등을 통해 실제 번역 품질을 반영하도록 고안된 지표
    * 주로 기계 번역에서 모델이 원문을 얼마나 정확히 번역했는지 평가할 때 사용


#### BLEU 평가

- 각 n-gram별로 **(생성 문장에 등장하는 n-gram 중 참조와 일치하는 횟수) / (생성 문장 전체 n-gram 수)**를 구해 정밀도를 계산

- 1,2,3,4-gram 정밀도를 기하평균으로 합산하고, Brevity Penalty를 곱해 최종 BLEU 스코어가 완성

- 생성 문장이 짧을수록 n-gram 개수가 적어 정밀도가 높아지는 문제 때문에 Brevity Penalty(지수 감소)를 적용하여 패널티를 부여

- 문장이 짧아 3-gram,4-gram 에서 불일치가 많을 때, BLEU 스코어가 0으로 가는 현상을 완화하기 위해 스무딩 기법(SmoothingFunction)을 적용.
    * `SmoothingFunction.method1` : 1-gram, 2-gram, ... n-gram 정밀도에 대해, 분모에 1을 더하고 분자에도 1을 더하는 방식(+1 smoothing).

In [ ]:
# NLTK(Natural Language Toolkit) 라이브러리를 가져옵니다. 자연어 처리를 위한 파이썬 표준 라이브러리입니다.
import nltk
# BLEU 점수 계산을 위한 함수들을 가져옵니다.
# corpus_bleu: 여러 문장(문서 전체)에 대한 BLEU 점수 계산
# sentence_bleu: 문장 하나에 대한 BLEU 점수 계산
# SmoothingFunction: 점수가 0이 되는 것을 방지하기 위한 보정 함수
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

# [전체 흐름]
# 이 코드는 기계 번역의 품질을 평가하는 가장 대표적인 지표인 **BLEU Score (Bilingual Evaluation Understudy Score)**를 계산합니다.
# 원리: "모델이 뱉은 문장(Hypothesis)"이 "정답 문장(Reference)"과 얼마나 겹치는지를 N-gram(단어 덩어리) 단위로 측정합니다.

# 1. 데이터 준비
# hypo (Hypothesis): 모델이 예측(생성)한 문장입니다. 토큰화된 리스트 형태여야 합니다.
# (주의: 여기서는 '먹었'으로 끝납니다.)
hypo  = ["나는", "저녁에", "밥을", "먹었", "."]

# ref (Reference): 정답(참조) 문장입니다.
# (주의: 정답은 여러 개일 수 있으므로, 리스트 안에 리스트가 들어가는 이중 리스트 구조여야 합니다.)
# (여기서는 '먹었다'로 끝납니다. hypo와 미묘하게 다릅니다.)
ref   = [["나는", "저녁에", "밥을", "먹었다", "."]]


# 2. N-gram 별 점수 계산
# weights=(w1, w2, w3, w4): 1~4 gram 점수에 각각 얼마의 가중치를 줄지 결정합니다.

# [1-gram] 단어 1개씩 쪼개서 비교 (Unigram)
# hypo(5개): "나는", "저녁에", "밥을", "먹었", "."
# ref: "나는", "저녁에", "밥을", "먹었다", "."
# 일치: "나는", "저녁에", "밥을", "." (4개 일치)
# 정밀도: 4 / 5 = 0.8
bleu_1gram = sentence_bleu(ref, hypo, weights=(1, 0, 0, 0))

# [2-gram] 단어 2개씩 묶어서 비교 (Bigram)
# hypo 뭉치(4개): ("나는", "저녁에"), ("저녁에", "밥을"), ("밥을", "먹었"), ("먹었", ".")
# ref 뭉치: ("나는", "저녁에"), ("저녁에", "밥을"), ("밥을", "먹었다"), ("먹었다", ".")
# 일치: ("나는", "저녁에"), ("저녁에", "밥을") -> 2개 일치
# 정밀도: 2 / 4 = 0.5
bleu_2gram = sentence_bleu(ref, hypo, weights=(0, 1, 0, 0))

# [3-gram] 단어 3개씩 묶어서 비교 (Trigram)
# hypo 뭉치(3개): ("나는","저녁에","밥을"), ("저녁에","밥을","먹었"), ("밥을","먹었",".")
# 일치: ("나는","저녁에","밥을") -> 1개 일치
# 정밀도: 1 / 3 = 0.333...
bleu_3gram = sentence_bleu(ref, hypo, weights=(0, 0, 1, 0))

# [4-gram] 단어 4개씩 묶어서 비교 (4-gram)
# hypo 뭉치(2개): ("나는","저녁에","밥을","먹었"), ("저녁에","밥을","먹었",".")
# 일치: 0개 (정답은 "먹었다"이므로 "먹었"이 들어간 뭉치는 다 틀림)
# 정밀도: 0 / 2 = 0.0
# (참고: BLEU는 수학적으로 기하평균을 쓰는데, 하나라도 0이면 전체가 0이 됩니다.
# NLTK는 완전한 0 대신 계산 오류를 막기 위해 '2.22...e-308' 같은 아주 작은 수(epsilon)를 반환합니다.)
bleu_4gram = sentence_bleu(ref, hypo, weights=(0, 0, 0, 1))

# [최종 BLEU Score]
# 보통 1~4 gram을 모두 25%씩 반영하여 기하평균을 냅니다.
# 식: exp(0.25*log(p1) + 0.25*log(p2) + 0.25*log(p3) + 0.25*log(p4))
# 문제점: 위에서 4-gram 정밀도가 '0'이 나왔기 때문에, 곱하기를 하면 전체 점수도 '0'(아주 작은 수)이 되어버립니다.
# (출력된 7.38...e-78은 사실상 0점이라는 뜻입니다.)
bleu_scroe = sentence_bleu(ref, hypo, weights=(0.25, 0.25, 0.25, 0.25))


# 결과 출력
print("BLEU score 1g:", bleu_1gram)
print("BLEU score 2g:", bleu_2gram)
print("BLEU score 3g:", bleu_3gram)
print("BLEU score 4g:", bleu_4gram)
print("BLEU score :", bleu_scroe)


# [Smoothing Function (평활화)]
# 4-gram이 하나도 안 맞아서 전체 점수가 0이 되는 억울한 상황을 막기 위한 기법입니다.
# 분자가 0일 때, 아주 작은 값(epsilon)을 더해줘서 0이 되는 것을 방지합니다.
# method1: 가장 기본적인 smoothing 방식
smooth = SmoothingFunction().method1

# Smoothing을 적용하여 다시 계산
# 이제 4-gram이 0이어도 전체 점수가 0으로 죽지 않고, 유의미한 점수(0.28...)가 나옵니다.
bleu_smooth = sentence_bleu(ref, hypo, smoothing_function=smooth)
print("BLEU score smooth:", bleu_smooth)

BLEU score 1g: 0.8
BLEU score 2g: 0.5
BLEU score 3g: 0.3333333333333333
BLEU score 4g: 2.2250738585072626e-308
BLEU score : 7.380245217279165e-78
BLEU score smooth: 0.28574404296988


In [ ]:
# [전체 흐름]
# 이 코드는 학습된 트랜스포머 모델을 사용하여 실제로 '대화(추론)'를 해보고,
# 그 성능을 'BLEU 점수'로 평가하는 과정입니다.
#
# 1. (단일 테스트): 임의의 입력으로 잘 작동하는지 한 번 확인합니다.
# 2. (평가 루프): 데이터셋에서 하나씩 꺼내어 모델에게 "대답해봐"라고 시키고(generate), 정답과 비교합니다.
# 3. (텍스트 변환): 숫자로 나온 결과를 사람이 읽을 수 있는 '한글'로 바꿉니다.
# 4. (점수 산출): 전체적인 번역(생성) 품질을 BLEU 점수로 계산합니다.

# --- 1. 단일 문장 생성 테스트 ---
# (이전 코드에서 정의된 'eco_inp'를 넣어봅니다.)
# 2=[BOS], 3=[EOS], 0=[PAD]
generated_token = transformer.generate(eco_inp, 2, 3, 0, device=device)

# (결과 출력)
# 최종 출력 형상: [1, 10, 16] (이건 forward 결과 shape)
print(f'최종 출력 형상 {out.shape}')
# 토큰 생성: [2, 6, 3] -> [BOS, (어떤 단어), EOS]
# (학습이 잘 안 된 초기 상태라면 이렇게 짧거나 엉뚱한 결과가 나올 수 있습니다.)
print(f'토큰 생성 {generated_token}')

최종 출력 형상 torch.Size([1, 10, 16])
토큰 생성 [2, 6, 3]


In [ ]:
# --- 2. 전체 평가 루프 (Inference Loop) ---

# (중요!) 추론(generate)은 한 번에 한 문장씩 순서대로 만드는 과정이므로,
# 배치 크기를 1로 설정하는 것이 일반적이고 구현하기 쉽습니다.
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# 결과를 저장할 리스트 (BLEU 점수 계산용)
all_preds = []  # 모델이 예측한 문장들
all_tars = []   # 실제 정답 문장들

# 모델을 '평가 모드'로 전환 (드롭아웃 비활성화 등)
transformer.eval()

# 평가 때는 역전파(기울기 계산)가 필요 없으므로 끔 (메모리 절약, 속도 향상)
with torch.no_grad():
    # 데이터로더에서 (질문, 답변) 쌍을 하나씩 꺼냅니다.
    for i, (inp, tar) in enumerate(dataloader):
        # 데이터를 GPU로 이동
        inp = inp.long().to(device)
        tar = tar.long().to(device)

        # [문장 생성]
        # 모델에게 질문(inp)을 주고, [BOS](시작)부터 [EOS](끝)까지 스스로 문장을 만들게 합니다.
        # (Autoregressive: 자기 회귀 생성)
        pred_ids = transformer.generate(inp, sp.bos_id(), sp.eos_id(), pad_token=pad_token, device=device)

        # 정답(tar)도 리스트 형태로 변환 (비교를 위해)
        target_ids = tar.detach().cpu().tolist()

        # [후처리: 토큰 정리]
        # 생성된 ID 리스트에서 [UNK](Unknown, 모르는 단어) 토큰을 제거합니다. (깔끔한 문장을 위해)
        pred_ids = [id for id in pred_ids if id != sp.unk_id()]
        # 정답 ID 리스트에서도 [UNK]를 제거합니다. (target_ids는 배치가 1이라 [0]으로 접근)
        target_ids = [id for id in target_ids[0] if id != sp.unk_id()]

        # [디코딩: 숫자 -> 문자]
        # 토크나이저(sp)를 이용해 ID 리스트를 다시 '한글 문장'으로 복원합니다.
        pred_tokens = sp.decode(pred_ids)   # 모델의 대답
        tar_tokens = sp.decode(target_ids)  # 실제 정답

        # (화면 출력: 잘 대답하고 있나 눈으로 확인)
        print(f'pred: {pred_tokens}') # 모델 예측
        print(f'tar: {tar_tokens} \n') # 실제 정답

        # [리스트에 저장]
        # BLEU 점수 계산을 위해 문자열을 리스트에 모읍니다.
        all_preds.append(pred_tokens)
        # (주의!) corpus_bleu는 정답(Reference)이 '여러 개'일 수 있다고 가정하므로,
        # 정답을 리스트로 한 번 더 감싸서 [[정답1], [정답2]...] 형태가 되게 넣어야 합니다.
        all_tars.append([tar_tokens])

        # (테스트용) 너무 오래 걸리니 12개(0~11)만 확인하고 멈춥니다.
        if i > 10:
            break

# --- 3. BLEU 점수 계산 ---
# 모아둔 예측값(all_preds)과 정답값(all_tars)을 비교하여 전체 평균 BLEU 점수를 계산합니다.
# (SmoothingFunction: 점수가 0이 되는 것을 방지하는 보정)
bleu_score = corpus_bleu(all_tars, all_preds, smoothing_function=smooth)
print(f'bleu_score:{bleu_score}')

# [결과 분석 및 한계점]
# 출력된 예시들을 보면 문맥은 어느 정도 맞지만, 완벽하게 똑같지는 않습니다.
# 예) pred: "학교 폭력을 당해서 학교 폭력을 당하셨군요." (말이 좀 꼬임)
#     tar: "학교 폭력을 보고 분노를 느끼셨군요."
#
# BLEU Score (0.21):
# BLEU는 '단어가 얼마나 정확히 겹치는지'를 봅니다.
# 하지만 챗봇같은 '생성' 문제에서는 단어가 달라도 뜻이 통하면 정답입니다.
# (예: "밥 먹었어?" vs "식사 하셨나요?" -> 뜻은 같지만 BLEU 점수는 낮음)
# 따라서 최근에는 단순 겹침(n-gram)보다 의미적 유사도를 보는 'BERTScore' 같은 지표를 더 많이 씁니다.

pred: 선생님이 사용자님을 몰라서 속상하셨군요.
tar: 선생님이 대답해 주지 않아 속상하셨군요. 

pred: 나이가 드는 왜 걱정이 되시나요?
tar: 최근 걱정할 일이 있으셨나요? 

pred: 친구와 여행을 가기로 하셨군요.
tar: 여행을 갈 계획이 있으시군요. 어떤 기분이신가요? 

pred: 아내가 임신하셨군요.
tar: 아내가 임신하셨군요! 

pred: 상담하고 상담을 받으셨군요.
tar: 그러셨군요. 주로 어떤 대화를 하고 오셨나요? 

pred: 가족을 많이 신경 써주시는군요.
tar: 어떤 방법으로 신경을 많이 쓰세요? 

pred: 낯선 사람들을 만나고 계시는군요.
tar: 살고 있는 동네에 만족감이 엄청 높으시네요. 주변 분들이 그렇게 좋으세요? 

pred: 남자친구와 다투시는 일이 있으셨군요.
tar: 남자친구와 수학여행 이야기를 하다가 의견이 맞지 않아 걱정되시는군요. 

pred: 왜 그렇게 생각하시나요?
tar: 짜증나시겠어요. 헌데 왜 때리는 건가요? 

pred: 약을 물어야 하는 약을 물어야 하는군요.
tar: 약을 많이 드셔야 해서 난처하시군요. 

pred: 남자친구가 멋진 남자 친구가 멋진다는 것을 알게 되어 좋으시군요.
tar: 예비 남편이 멋져서 감사한 마음이 드는군요. 

pred: 학교 폭력을 당해서 학교 폭력을 당하셨군요.
tar: 학교 폭력을 보고 분노를 느끼셨군요. 

bleu_score:0.21316968646492845


> 기계 번역이 아닌, 좀더 일반적인 자연어 생성 문제에서는 BLUE같은 n-gram Overlap 방식 보다 문장 유사도를 파악 할수 있는 신경망 임베딩 기반 지표가 많이 활용